# 🧠 Level 6 — Context Engineering (Mukammal Notebook)
### AI Engineering Roadmap 2026 · Phase: Agentic · Roman Urdu Edition

> **Maqsad:** Model ki mehdood context window ko *jaan boojh kar* manage karna — kya andar aaye, kya rahe, kya compress ho aur kya bahar rakha jaye — aur ek **agent harness** banana jo lambe kaam mein yeh faislay khud laagu kare.

| | |
|---|---|
| **Prerequisite** | Level 5 · AI Agents (tool calling loop, memory basics) |
| **Interview weight** | 6/10 |
| **Job weight** | 8/10 |
| **LLM provider** | Groq (real API — koi fake/mock nahi) |

**Is notebook ka tareeqa:** har module mein pehle **theory** (Roman Urdu), phir **code** (real Groq calls), phir **interview angle**. Aakhir mein **Capstone** (naive vs managed agent ka naap tol ke muqabla) aur **22 interview sawal-jawab**.

### 📚 Fehrist (Table of Contents)
| # | Module | Kya banayenge |
|---|---|---|
| 0 | Setup | Groq client, token counter, usage logger, tool loop |
| 6.1 | The Context Window as a Resource | Context inspector, context-rot experiment, conflicting context demo, budget allocator |
| 6.2 | Context Selection & Assembly | Scored `ContextBuilder` (always / retrieved / excluded), stale-context cutter |
| 6.3 | Compression & Compaction | Structured summarizer, pinned exact facts, rolling + hierarchical summary, info-loss eval |
| 6.4 | Memory & State Offloading | Scratchpad + note-taking agent jo context reset ke baad bhi yaad rakhe |
| 6.5 | Long-Context Strategies | Stuffing vs Retrieval vs Map-Reduce vs Reorder — naap kar muqabla |
| 6.6 | Tool Context Management | 40 tools ka token bojh, dynamic tool exposure, result handles, observation masking |
| 6.7 | Context Caching & Efficiency | Prefix-cache experiment (good vs bad structure), cost calculator |
| 6.8 | Harness Engineering | Mini coding-agent harness: AGENTS.md, task list, progress file, tests-gated "done", sub-agent |
| 🚀 | Capstone | Context-Managed Long-Horizon Agent vs Naive Agent |
| 🎤 | Interview Prep | 22 sawal + model jawab, cheat sheet |

> 💡 **Tip:** Notebook ko upar se neeche tarteeb se chalayein. Har section pichle helpers par khara hai.

---
# 0️⃣ Setup

### Windows par install (ek dafa)
Terminal / PowerShell (ya VS Code terminal) mein:
```
pip install groq tiktoken numpy scikit-learn matplotlib python-dotenv
```

### Groq API key
1. https://console.groq.com/keys se key banayein (`gsk_...`).
2. Teen mein se koi ek tareeqa:
   - **`.env` file** (recommended): notebook wale folder mein `.env` banayein aur likhein `GROQ_API_KEY=gsk_...`
   - **PowerShell:** `setx GROQ_API_KEY "gsk_..."` phir VS Code/Jupyter **restart** karein
   - Kuch na karein — neeche wala cell `getpass` se poochh lega (key screen par nazar nahi aayegi)

> ⚠️ Key kabhi notebook ke andar hard-code na karein (L2 · API Key Safety). Notebook GitHub par jaye to key leak ho jati hai.

### Models ke baare mein (Oct 2026)
Groq ne free/developer tier par Llama 3.1 8B aur Llama 3.3 70B band kar diye hain. Ab production text models **`openai/gpt-oss-20b`** (bohot tez, sasta) aur **`openai/gpt-oss-120b`** (zyada qabil) hain. Dono **reasoning models** hain — jawab se pehle andar "sochte" hain, aur woh soch **output tokens** mein count hoti hai (L1.7 yaad karein). Is liye:
- `max_tokens` kabhi bohot kam na rakhein (warna sara budget soch mein khatam aur `content` khali!)
- `reasoning_effort="low"` rakhenge taake tez aur sasta rahe.

Neeche wala code khud dekhta hai ke aap ke account par kaunse models available hain.

In [ ]:
# !pip install -q groq tiktoken numpy scikit-learn matplotlib python-dotenv   # zaroorat ho to uncomment karein

import os, sys, time, json, re, math, random, hashlib, textwrap, subprocess, shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt

from groq import Groq, RateLimitError, BadRequestError, APIStatusError, APIConnectionError

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if not os.environ.get("GROQ_API_KEY"):
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Groq API key paste karein (gsk_...): ")

# max_retries=0 → retries hum khud handle karenge (L2.9: exponential backoff + jitter)
client = Groq(max_retries=0, timeout=120)

def pick_models():
    """Account par dastiyab models dekh kar FAST aur SMART model chunna (hard-code nahi — L1.10 Versions & Deprecation)."""
    try:
        available = sorted(m.id for m in client.models.list().data)
    except Exception as e:
        print("⚠️ models.list() fail hua:", e)
        available = []
    skip = ("whisper", "tts", "guard", "orpheus", "compound", "playai")
    text_models = [m for m in available if not any(s in m.lower() for s in skip)]
    fast = next((m for m in ["openai/gpt-oss-20b", "openai/gpt-oss-120b"] if m in available),
                text_models[0] if text_models else "openai/gpt-oss-20b")
    smart = next((m for m in ["openai/gpt-oss-120b", "openai/gpt-oss-20b"] if m in available), fast)
    return fast, smart, available

FAST_MODEL, SMART_MODEL, AVAILABLE = pick_models()
print("Available text models:", [m for m in AVAILABLE if "whisper" not in m])
print("FAST_MODEL  →", FAST_MODEL)
print("SMART_MODEL →", SMART_MODEL)

### Token counter
Groq ke gpt-oss models `o200k`-family tokenizer istemal karte hain, is liye `tiktoken` ka `o200k_base` acha **andaza** deta hai. Asal number hamesha API ke `usage.prompt_tokens` se aata hai — hum dono dekhenge.

> 🎯 **Interview point:** "Local token count andaza hai, billing ka sach `usage` field hai. Har model ka tokenizer alag hota hai."

In [ ]:
try:
    import tiktoken
    _ENC = tiktoken.get_encoding("o200k_base")
    def count_tokens(text: str) -> int:
        return len(_ENC.encode(text or ""))
    TOKENIZER = "tiktoken o200k_base"
except Exception as e:  # offline ho ya tiktoken na ho to mota andaza
    def count_tokens(text: str) -> int:
        return max(1, len(text or "") // 4)
    TOKENIZER = f"fallback len/4 ({type(e).__name__})"

def count_msg_tokens(messages) -> int:
    """Chat messages ka andaza: har message ka content + ~4 tokens role/formatting overhead."""
    total = 0
    for m in messages:
        c = m.get("content") if isinstance(m, dict) else getattr(m, "content", "")
        total += count_tokens(c if isinstance(c, str) else json.dumps(c or "")) + 4
        tcs = m.get("tool_calls") if isinstance(m, dict) else None
        if tcs:
            total += count_tokens(json.dumps(tcs))
    return total

print("Tokenizer:", TOKENIZER)
for s in ["Hello world", "Assalam o alaikum, aap ka order kab tak pohanchega?", "ORD-10293 Rs 45,500 🚚"]:
    print(f"{count_tokens(s):>3} tokens | {s}")

### `llm()` helper — har call ka hisaab
Yeh ek hi function poore notebook mein istemal hoga. Yeh L2 ke sabaq laagu karta hai:
- **Rate limit (429)** par `retry-after` header ya exponential backoff + jitter
- `reasoning_effort` support na ho to khud hata deta hai
- Har call ka **prompt tokens, output tokens, cached tokens, latency** `USAGE_LOG` mein — taake hum har experiment *naap* sakein (sirf "lagta hai behtar hai" nahi)

In [ ]:
USAGE_LOG = []

# USD per 1M tokens (input, output). Qeematein badalti hain → console.groq.com/pricing se check karein.
PRICES = {"openai/gpt-oss-20b": (0.075, 0.30), "openai/gpt-oss-120b": (0.15, 0.60)}

def _cached_tokens(usage) -> int:
    d = getattr(usage, "prompt_tokens_details", None)
    if d is None:
        return 0
    if isinstance(d, dict):
        return d.get("cached_tokens") or 0
    return getattr(d, "cached_tokens", 0) or 0

def _retry_after(err, attempt):
    try:
        ra = err.response.headers.get("retry-after")
        if ra:
            return float(ra) + 0.5
    except Exception:
        pass
    return min(60, 2 ** attempt) + random.random()

def llm(messages, model=None, max_tokens=1500, temperature=0.2, tools=None, tool_choice=None,
        json_mode=False, reasoning_effort="low", tag="", max_retries=7):
    """Ek Groq chat call. Return: dict(text, message, usage, finish_reason)."""
    model = model or FAST_MODEL
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    kwargs = dict(model=model, messages=messages, max_completion_tokens=max_tokens, temperature=temperature)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice or "auto"
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    if reasoning_effort and "gpt-oss" in model:
        kwargs["reasoning_effort"] = reasoning_effort

    resp, t0 = None, time.perf_counter()
    for attempt in range(max_retries):
        t0 = time.perf_counter()
        try:
            resp = client.chat.completions.create(**kwargs)
            break
        except RateLimitError as e:
            wait = _retry_after(e, attempt)
            print(f"   ⏳ rate limit — {wait:.1f}s intezar (attempt {attempt + 1})")
            time.sleep(wait)
        except BadRequestError as e:
            msg = str(e).lower()
            if "reasoning_effort" in kwargs and "reasoning" in msg:
                kwargs.pop("reasoning_effort"); continue
            if "json_validate_failed" in msg or "failed to generate json" in msg:
                time.sleep(0.5); continue           # model ne invalid JSON diya → dobara
            if "tool_use_failed" in msg or "failed to call a function" in msg:
                time.sleep(0.5); continue           # model ne kharab tool call banayi → dobara
            raise
        except (APIConnectionError, APIStatusError) as e:
            status = getattr(e, "status_code", 500) or 500
            if status == 413:
                raise RuntimeError("413 Request too large — yeh ek request aap ke tier ki tokens-per-minute had se bari hai. "
                                   "Context chhota karein (yahi to L6 ka sabaq hai!) ya max_tokens kam karein.") from e
            if status < 500 or attempt == max_retries - 1:
                raise
            time.sleep(min(30, 2 ** attempt) + random.random())
    if resp is None:
        raise RuntimeError("Retries khatam — rate limit ya network masla.")

    latency = time.perf_counter() - t0
    choice = resp.choices[0]
    u = resp.usage
    rec = dict(tag=tag, model=model, prompt_tokens=u.prompt_tokens, completion_tokens=u.completion_tokens,
               cached_tokens=_cached_tokens(u), latency=round(latency, 3))
    USAGE_LOG.append(rec)
    text = choice.message.content or ""
    if not text and not getattr(choice.message, "tool_calls", None) and choice.finish_reason == "length":
        print("   ⚠️ content khali + finish_reason=length → reasoning ne max_tokens kha liye, max_tokens barhayein")
    return {"text": text, "message": choice.message, "usage": rec, "finish_reason": choice.finish_reason}

def ask(prompt, system=None, **kw) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    return llm(msgs, **kw)["text"]

def cost_usd(rec, cached_discount=0.5):
    """Andazan kharcha. cached_discount = cached input tokens par chhoot (provider ke docs check karein)."""
    pin, pout = PRICES.get(rec["model"], (0.15, 0.60))
    fresh = rec["prompt_tokens"] - rec["cached_tokens"]
    return (fresh * pin + rec["cached_tokens"] * pin * (1 - cached_discount) + rec["completion_tokens"] * pout) / 1e6

def usage_summary(tag_prefix=""):
    rows = [r for r in USAGE_LOG if r["tag"].startswith(tag_prefix)]
    if not rows:
        return {}
    return dict(calls=len(rows),
                prompt_tokens=sum(r["prompt_tokens"] for r in rows),
                completion_tokens=sum(r["completion_tokens"] for r in rows),
                cached_tokens=sum(r["cached_tokens"] for r in rows),
                peak_prompt=max(r["prompt_tokens"] for r in rows),
                latency_s=round(sum(r["latency"] for r in rows), 2),
                cost_usd=round(sum(cost_usd(r) for r in rows), 6))

def wrap(t, w=110):
    return "\n".join(textwrap.fill(line, w) if line.strip() else "" for line in str(t).splitlines())

# Smoke test — asli Groq call
r = llm("Ek line mein batao: context window kya hai?", tag="setup")
print(wrap(r["text"]))
print(r["usage"])

### Tool-calling loop (L2 / L5 se) — reusable
Context engineering zyada tar **agents** ke liye hai, is liye ek generic loop bana lete hain. Is mein ek **`on_step` hook** hai — har step ke baad hum yahan context ko manage karenge (masking, compaction, state injection). Yahi "harness" ka bunyadi darwaza hai.

In [ ]:
def tool_schema(name, description, properties=None, required=None):
    """OpenAI/Groq tool definition banane ka chhota helper."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties or {}, "required": required or []}}}

def call_tool(registry, name, arguments):
    """Tool chalao — model ke arguments ko UNTRUSTED samjho (L2.6, L8.4)."""
    fn = registry.get(name)
    if fn is None:
        return json.dumps({"error": f"unknown tool '{name}'. Available: {list(registry)}"})
    try:
        args = json.loads(arguments or "{}")
        if not isinstance(args, dict):
            raise ValueError("arguments object hone chahiye")
    except Exception as e:
        return json.dumps({"error": f"bad JSON arguments: {e}"})
    try:
        out = fn(**args)
    except TypeError as e:
        return json.dumps({"error": f"wrong arguments: {e}"})
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})
    return out if isinstance(out, str) else json.dumps(out, ensure_ascii=False)

def run_tool_loop(messages, tools, registry, model=None, max_steps=10, on_step=None, tag="agent",
                  verbose=True, **kw):
    """Goal → decide → act → observe → repeat. Return: (final_text, messages, steps)."""
    for step in range(1, max_steps + 1):
        r = llm(messages, model=model, tools=tools, tag=tag, **kw)
        m = r["message"]
        tcs = getattr(m, "tool_calls", None) or []
        asst = {"role": "assistant", "content": m.content if tcs else (m.content or "")}
        if tcs:
            asst["tool_calls"] = [{"id": tc.id, "type": "function",
                                   "function": {"name": tc.function.name,
                                                "arguments": tc.function.arguments or "{}"}} for tc in tcs]
        messages.append(asst)
        if not tcs:
            return r["text"], messages, step
        for tc in tcs:
            result = call_tool(registry, tc.function.name, tc.function.arguments)
            if verbose:
                print(f"   🔧 step {step}: {tc.function.name}({(tc.function.arguments or '')[:80]}) → {len(result)} chars")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        if on_step:
            stop = on_step(messages, step)
            if stop:
                return stop if isinstance(stop, str) else "[stopped by harness]", messages, step
    return "[max_steps reached]", messages, max_steps

print("✅ tool loop ready")

### Shared demo data — ek online watch store ("TimeCraft PK")
Poore notebook mein ek hi domain istemal karenge taake har technique ka asar saaf nazar aaye: policies, products, orders aur support tickets. Data **deterministic** (fixed seed) hai taake ground truth humein pata ho aur hum accuracy naap sakein.

In [ ]:
CITIES = ["Lahore", "Karachi", "Islamabad", "Multan", "Faisalabad"]
STATUSES = ["DELIVERED", "SHIPPED", "DELAYED", "CANCELLED", "PROCESSING"]
PRODUCTS = [
    {"sku": "TC-AV-01", "name": "Aviator Chrono", "price": 18500, "warranty_months": 12},
    {"sku": "TC-DV-02", "name": "Diver Pro 200m", "price": 26500, "warranty_months": 24},
    {"sku": "TC-CL-03", "name": "Classic Leather", "price": 9500, "warranty_months": 6},
    {"sku": "TC-SM-04", "name": "Smart Fit X", "price": 14000, "warranty_months": 12},
    {"sku": "TC-WL-05", "name": "Slim Wallet Combo", "price": 6500, "warranty_months": 3},
]
NOTE_SNIPPETS = ["Customer ne gift wrap maanga", "Call karne se pehle WhatsApp karein", "Office address par deliver",
                 "COD — change tayyar rakhein", "Doosri dafa order", "Weekend delivery preferred"]

def make_orders(n, seed=0):
    rng = random.Random(seed)
    out = []
    for i in range(n):
        p = rng.choice(PRODUCTS)
        out.append({"order_id": f"ORD-{1000 + i}", "city": rng.choice(CITIES), "status": rng.choice(STATUSES),
                    "item": p["name"], "amount_pkr": p["price"] * rng.choice([1, 1, 1, 2]),
                    "note": rng.choice(NOTE_SNIPPETS)})
    return out

POLICY_DOCS = [
    {"id": "pol-returns-2024", "title": "Returns FAQ (purana blog post)", "date": "2024-03-01", "source": "blog",
     "text": "Returns: customer delivery ke 7 din ke andar item wapas kar sakta hai. Return shipping customer dega."},
    {"id": "pol-returns-2026", "title": "Official Returns Policy v3", "date": "2026-07-15", "source": "official_policy",
     "text": "Returns: delivery ke 14 din ke andar return ho sakta hai, item unused aur original box mein ho. "
             "Defective item par return shipping store dega. Refund 5 working days mein bank transfer se."},
    {"id": "pol-shipping", "title": "Shipping Policy", "date": "2026-05-01", "source": "official_policy",
     "text": "Lahore, Karachi, Islamabad: 2-3 working days. Baqi shehar: 4-6 working days. "
             "Rs 10,000 se zyada ke order par free shipping, warna Rs 250. Sirf Cash on Delivery (COD)."},
    {"id": "pol-warranty", "title": "Warranty Policy", "date": "2026-01-10", "source": "official_policy",
     "text": "Warranty sirf manufacturing defect cover karti hai; paani ka nuqsan sirf Diver series par covered hai. "
             "Claim ke liye order ID aur tasveer WhatsApp par bhejein."},
    {"id": "pol-exchange", "title": "Exchange Policy", "date": "2026-02-20", "source": "official_policy",
     "text": "Exchange 10 din ke andar ek dafa free. Doosri exchange par Rs 300 charge."},
]
ORDERS = make_orders(60, seed=7)
print(len(ORDERS), "orders | sample:", ORDERS[0])

---
# 6.1 · The Context Window as a Resource
*"Context ek budget hai."*

## 📖 Theory

### Context Engineering kya hai?
**Context Engineering** = model ko **har qadam par** behtareen, chhota aur saaf context dene ka fan. Model sirf wohi "jaanta" hai jo us waqt uski window mein hai (training knowledge ke ilawa). Is liye sawal yeh nahi ke "prompt kaisa likhun" — sawal yeh hai ke **"is waqt, is faislay ke liye, window mein kya hona chahiye?"**

| Prompt Engineering (L3) | Context Engineering (L6) |
|---|---|
| Ek achhi **hidayat** likhna | Poori **window** ko design karna — instructions + history + retrieved docs + memory + tool definitions + tool results + state |
| Aksar static, ek dafa likha | **Dynamic** — har step par dobara assemble hota hai |
| Single call ka masla | Lambe, multi-step **agent runs** ka masla |
| "Kaise kahun?" | "Kya dikhaun, kya chhupaun, kya yaad rakhun?" |

> Prompt engineering, context engineering ka **ek hissa** hai — system prompt bhi context ka ek tukra hai.

### Context as a Budget
"Zyada context = behtar jawab" **ghalat** hai. Har token ki teen qeematein hain:
1. **Paisa** — input tokens par bill (har request par!).
2. **Latency** — zyada prefill = zyada TTFT.
3. **Attention** — transformer mein har token baqi sab se "muqabla" karta hai. Jitne zyada tokens, utna har aham token ka hissa kam. Ise **attention budget** kehte hain.

### Context Rot
Context lamba hone par model ki performance ka **dheere dheere girna** — chahe window abhi bhari na ho. 128k window wala model 100k par woh kaam bura karta hai jo 5k par acha karta tha. Research (jaise Chroma ki 2025 "Context Rot" report) ne dikhaya ke simple tasks par bhi input lambai barhne se accuracy girti hai — khaas kar jab **distractors** (milti julti magar ghalat maloomat) hon ya kaam ko kayi jagah se maloomat **jorni** pare.

### Distraction & Conflicting Context — 4 failure modes (interview favorite)
| Failure | Matlab | Misaal | Ilaj |
|---|---|---|---|
| **Context Poisoning** | Ghalat baat (hallucination/ghalat tool result) context mein aa gayi aur baar baar refer ho rahi hai | Agent ne ek dafa ghalat order ID likha, ab har step usi par | Validation, ghalat state ko hatana, fresh context |
| **Context Distraction** | Itni lambi history ke model naya sochne ke bajaye purane actions dohrata hai | 50 steps baad agent wahi 3 tools loop mein | Compaction, summarization |
| **Context Confusion** | Faltu cheezen (bohot tools, ghair mutalliqa docs) model ko bhatkati hain | 40 tools mein se ghalat tool chunna | Selection, dynamic tool exposure |
| **Context Clash** | Context ke do hisse ek doosre ke khilaf | Purana FAQ: 7 din return; naya policy: 14 din | Dates/source label, purana hatao, priority rule |

### Inspecting the Window
Debugging ka pehla usool: **dekho model tak asal mein kya pohancha.** Aksar "model bewaqoof hai" ka asal sabab hota hai: document truncate ho gaya, history mein purani hidayat reh gayi, tool result 20k tokens ka tha, ya system prompt hi nahi gaya.

### Budget Allocation
Har component ke liye mota hissa tay karein (misaal 32k window):
```
System instructions  ~ 5-10%   (stable, cacheable)
Tool definitions     ~ 5-15%   (sirf mutalliqa tools)
Memory / profile     ~ 5%
Retrieved knowledge  ~ 25-40%
Conversation history ~ 15-25%  (purana hissa summary)
Working state        ~ 5%
Output reserve       ~ 15-20%  (reasoning models ke liye zyada!)
```

## 💻 Code 1 — Context Inspector
Ek chhota tool jo batata hai window ka kaunsa hissa kitna bhara hai — aur hamara andaza asli `usage.prompt_tokens` se kitna milta hai.

In [ ]:
def inspect_context(components: dict, window: int = 32_000, reserve_output: int = 4_000, title="Context window"):
    """components: {"naam": text} → har hisse ke tokens, % aur bacha hua budget."""
    rows = [(k, count_tokens(v)) for k, v in components.items()]
    used = sum(t for _, t in rows)
    print(f"{title}  (window={window:,}, output reserve={reserve_output:,})")
    print(f"{'component':<24}{'tokens':>8}{'% window':>10}")
    for k, t in rows:
        print(f"{k:<24}{t:>8,}{t / window:>10.1%}  " + "█" * max(1, int(40 * t / window)))
    free = window - used - reserve_output
    print(f"{'TOTAL':<24}{used:>8,}{used / window:>10.1%}   | bacha hua input budget: {free:,}")
    if free < 0:
        print("🚨 OVERFLOW — truncation hogi ya API error. Kuch compress/hatayein!")
    return rows

system_prompt = ("Tum TimeCraft PK ke support assistant ho. Sirf diye gaye policy docs se jawab do. "
                 "Agar maloomat na ho to kaho 'mujhe yeh maloom nahi, insaan agent se baat karwata hoon'.")
history = "\n".join(f"user: order {o['order_id']} ka status?\nassistant: {o['status']}" for o in ORDERS[:25])
docs = "\n\n".join(f"[{d['id']}] {d['text']}" for d in POLICY_DOCS)
orders_dump = json.dumps(ORDERS, ensure_ascii=False)
question = "Mera Diver Pro paani mein kharab ho gaya, warranty mein aayega?"

parts = {"system": system_prompt, "history (25 turns)": history, "policy docs": docs,
         "orders tool result": orders_dump, "user question": question}
_ = inspect_context(parts, window=8_000, reserve_output=2_000, title="Naive context")

# Asal number API se:
msgs = [{"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{history}\n\n<docs>{docs}</docs>\n<orders>{orders_dump}</orders>\n\n{question}"}]
r = llm(msgs, tag="6.1-inspect")
print("\nLocal estimate:", count_msg_tokens(msgs), "| API prompt_tokens:", r["usage"]["prompt_tokens"])
print("Jawab:", wrap(r["text"])[:400])

**Sabaq:** sawal warranty ka tha, magar window ka bara hissa 60 orders ka JSON aur 25 purane turns kha gaye — jinka is sawal se koi taluq nahi. Yahi **context confusion/dilution** ki jarr hai. 6.2 mein isi ko theek karenge.

## 💻 Code 2 — Context Rot Experiment (asli naap)
Simple "needle in a haystack" aaj kal ke models aksar pass kar lete hain. Asli rot tab dikhta hai jab model ko **bohot si jagahon se maloomat jor kar** jawab dena ho. Experiment:
- N order records context mein (N = 10, 40, 120)
- Sawal: *"Lahore mein kitne orders DELAYED hain?"* — jawab humein code se pata hai (ground truth)
- Har N par 3 dafa poochte hain aur accuracy + tokens naapte hain

> Free tier par rate limit lag sakti hai — `llm()` khud intezar kar lega. Agar jaldi ho to `SIZES` chhota kar dein.

In [ ]:
def orders_as_text(orders):
    return "\n".join(f"{o['order_id']} | city={o['city']} | status={o['status']} | item={o['item']} | "
                     f"amount=Rs {o['amount_pkr']:,} | note={o['note']}" for o in orders)

SIZES, TRIALS = [10, 40, 120], 3
rot_results = []
for n in SIZES:
    correct, toks = 0, []
    for t in range(TRIALS):
        orders = make_orders(n, seed=100 + t)
        truth = sum(1 for o in orders if o["city"] == "Lahore" and o["status"] == "DELAYED")
        prompt = (f"Neeche order records hain.\n\n{orders_as_text(orders)}\n\n"
                  "Sawal: Lahore mein kitne orders ka status DELAYED hai? "
                  "Sirf aakhri line mein 'ANSWER: <number>' likho.")
        r = llm(prompt, tag="6.1-rot", max_tokens=2500)
        m = re.search(r"ANSWER:\s*(\d+)", r["text"])
        pred = int(m.group(1)) if m else -1
        correct += int(pred == truth)
        toks.append(r["usage"]["prompt_tokens"])
        print(f"N={n:<4} trial={t}  truth={truth:<3} pred={pred:<3} {'✅' if pred == truth else '❌'}  tokens={toks[-1]}")
    rot_results.append((n, correct / TRIALS, int(np.mean(toks))))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot([r[2] for r in rot_results], [r[1] for r in rot_results], marker="o")
for n, acc, tk in rot_results:
    ax.annotate(f"N={n}", (tk, acc), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("prompt tokens"); ax.set_ylabel("accuracy"); ax.set_ylim(-0.05, 1.05)
ax.set_title(f"Context rot — aggregation task ({FAST_MODEL})"); plt.tight_layout(); plt.show()

**Kya dekhna hai:** jaise jaise records (aur tokens) barhte hain, ginti ki ghaltiyan barhti hain — window abhi bohot khali hai, phir bhi. Reasoning models (gpt-oss) is mein behtar hain magar mehngi soch ke saath (completion tokens dekhein). **Ilaj context engineering hai, bari window nahi:** filter karo (sirf Lahore records), ya ginti **code** se karwao (tool), model se nahi.

## 💻 Code 3 — Context Clash (conflicting context)
Do policies — purana blog (7 din) aur naya official (14 din). Pehle baghair label ke, phir **date + source + priority rule** ke saath.

In [ ]:
q = "Mujhe delivery ke 10 din ho gaye hain. Kya main watch return kar sakta hoon? Ek line mein jawab do."
returns = [d for d in POLICY_DOCS if d["id"].startswith("pol-returns")]

# (A) Bina label — dono texts bas chipka diye
ctx_a = "\n".join(d["text"] for d in returns)
ans_a = ask(f"Maloomat:\n{ctx_a}\n\nSawal: {q}", tag="6.1-clash")

# (B) Labeled + conflict rule
ctx_b = "\n".join(f'<doc id="{d["id"]}" date="{d["date"]}" source="{d["source"]}">{d["text"]}</doc>' for d in returns)
rule = ("Agar documents aapas mein takrayein to: (1) source='official_policy' ko 'blog' par tarjeeh do, "
        "(2) phir nayi date ko. Jawab mein jis doc ka istemal kiya uski id likho.")
ans_b = ask(f"{ctx_b}\n\nSawal: {q}", system=rule, tag="6.1-clash")

print("A (bina label):", wrap(ans_a))
print("\nB (label + rule):", wrap(ans_b))

**Production sabaq:** clash ka sab se acha ilaj yeh hai ke **purana doc index se hi hata do** (6.2 · Cutting Stale Context). Rule likhna doosri line of defense hai.

## 💻 Code 4 — Budget Allocator
Har component ke liye budget tay karna aur overflow par batana ke **kis** hisse ko kaatna hai (priority ke hisaab se).

In [ ]:
@dataclass
class ContextBudget:
    window: int
    shares: dict                                     # component -> fraction of window
    priority: list                                   # pehle kaun kata jaye (low priority pehle)

    def plan(self, actual: dict):
        """actual: component -> tokens. Return: har component ki limit aur overflow advice."""
        limits = {k: int(self.window * f) for k, f in self.shares.items()}
        report, over = [], []
        for k, lim in limits.items():
            used = actual.get(k, 0)
            flag = "OK" if used <= lim else f"OVER by {used - lim:,}"
            report.append((k, used, lim, flag))
            if used > lim:
                over.append(k)
        print(f"{'component':<16}{'used':>8}{'limit':>8}  status")
        for k, u, l, f in report:
            print(f"{k:<16}{u:>8,}{l:>8,}  {f}")
        if over:
            order = [c for c in self.priority if c in over]
            print("➡️  Kaatne ki tarteeb:", order, "→ history ko summarize, retrieval ka top-k kam, tool output compact")
        return limits

budget = ContextBudget(
    window=32_000,
    shares={"system": 0.08, "tools": 0.10, "memory": 0.05, "retrieved": 0.35, "history": 0.20, "state": 0.04, "output": 0.18},
    priority=["history", "retrieved", "tools", "memory", "state", "system", "output"])
_ = budget.plan({"system": 900, "tools": 5200, "memory": 400, "retrieved": 9000, "history": 11_000, "state": 300})

### 🎤 Interview angle (6.1)
- *"1M token window hai to context engineering kyun?"* → Cost/latency har request par, attention budget, context rot, lost-in-the-middle. Bari window **gunjaish** hai, **hal** nahi.
- *"Quality gir gayi, pehla qadam?"* → **Window inspect karo** — log karo ke asal mein model ko kya bheja gaya. Aksar masla model ka nahi, input ka hai.
- Chaar failure modes (poisoning, distraction, confusion, clash) naam aur misaal ke saath yaad rakhein.

✅ **Outcome:** Pehchanna ke model ki kharabi asal mein context-management ka masla hai.

---
# 6.2 · Context Selection & Assembly
*"Sahi cheezen, sahi tarteeb."*

## 📖 Theory

### 4 bunyadi strategies — WRITE · SELECT · COMPRESS · ISOLATE
Poori Level 6 in chaar lafzon mein aa jati hai (interview mein yeh framework bata dein to strong impression):
| Strategy | Matlab | Module |
|---|---|---|
| **Write** | Maloomat ko context se **bahar** save karna (notes, files, DB) | 6.4 |
| **Select** | Har step par sirf **mutalliqa** cheezen andar laana | 6.2, 6.5, 6.6 |
| **Compress** | Jo andar hai use **chhota** karna (summary, compaction, trimming) | 6.3 |
| **Isolate** | Kaam ko alag contexts mein **baantna** (sub-agents, sandboxes, state fields) | 6.8 |

### Assembling the Window
Har request par window in tukron se banti hai: instructions, conversation history, retrieved knowledge, memory, tool definitions/results, app state. Assembly ek **function** honi chahiye (code), "jo mila chipka do" nahi.

### Always / Retrieved / Summarized / Excluded
| Category | Misaal |
|---|---|
| **Always** (pinned) | System rules, user ka naam/zuban, safety rules, current goal |
| **Retrieved** (zaroorat par) | Is sawal se mutalliqa policy docs, order record |
| **Summarized** | Purani conversation, lambe tool outputs |
| **Excluded** | Ghair mutalliqa docs, superseded/purani maloomat, dohrai hui cheezen, secrets |

### Prioritization
Har item ka score: `score = w1·relevance + w2·importance + w3·freshness + w4·trust`. Phir **greedy packing**: sab se zyada score wale items budget bharne tak.

### Distinguishable Ordering
Model ko pata hona chahiye kaun si cheez **hidayat** hai, kaun **saboot**, kaun **user ka data**. XML-style tags/sections istemal karein (L3.1, L8.2 — untrusted data alag). Aam tarteeb:
```
<instructions>      stable  ┐
<tools/policies>    stable  ├─ cache-friendly prefix (6.7)
<memory/profile>    semi    ┘
<retrieved_docs>    dynamic
<conversation_summary>
<recent_turns>
<task_state>
<user_question>     ← hamesha aakhir mein (recency ka faida)
```

### Cutting Stale Context
Dohra (duplicate), purana (superseded), takrata (conflicting) ya ghair mutalliqa content **nikalna**. Har item ke saath `date`, `source`, `supersedes` metadata rakhein taake code yeh faisla kar sake.

## 💻 Code — Scored `ContextBuilder`
Relevance ke liye TF-IDF cosine (sirf seekhne ke liye, koi extra model nahi chahiye). Production mein yahan embeddings + reranker (L4.3, L4.10) lagayein — builder ka design wahi rehta hai. Builder:
1. stale/superseded items hatata hai, 2. near-duplicates hatata hai, 3. score karta hai, 4. budget mein greedy pack karta hai, 5. tagged sections mein render karta hai.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import date

@dataclass
class ContextItem:
    id: str
    kind: str                 # "policy" | "memory" | "history" | "order" | ...
    text: str
    importance: float = 0.5   # 0..1 (business ahmiyat)
    trust: float = 0.5        # 0..1 (official=1, blog=0.4, user-said=0.5)
    when: str = "2026-01-01"  # ISO date
    pinned: bool = False      # "always" bucket
    supersedes: list = field(default_factory=list)

ROMAN_URDU_STOPWORDS = ("main mein hoon ka ke ki hai hain se par aur to kya kitne kitna ko ya bhi ek jo is us "
                        "yeh woh na nahi karun karo kar").split()

class ContextBuilder:
    def __init__(self, budget_tokens=1500, weights=(0.55, 0.2, 0.1, 0.15), today="2026-10-22",
                 min_relevance=0.1, max_per_kind=None):
        self.budget, self.w, self.today = budget_tokens, weights, date.fromisoformat(today)
        self.min_rel, self.max_per_kind = min_relevance, max_per_kind or {}
        self.log = []

    def _freshness(self, when):
        days = (self.today - date.fromisoformat(when)).days
        return math.exp(-days / 365)              # ~1 saal mein 1/e

    def build(self, query, items):
        self.log = []
        # 1) superseded hatao
        dead = {s for it in items for s in it.supersedes}
        live = [it for it in items if it.id not in dead]
        self.log += [f"EXCLUDED (superseded): {it.id}" for it in items if it.id in dead]
        # 2) near-duplicates hatao
        vec = TfidfVectorizer(stop_words=ROMAN_URDU_STOPWORDS).fit([query] + [it.text for it in live])
        M = vec.transform([it.text for it in live])
        sim = cosine_similarity(M)
        keep = []
        for i, it in enumerate(live):
            dup = next((j for j in keep if sim[i, j] > 0.9), None)
            if dup is not None:
                self.log.append(f"EXCLUDED (duplicate of {live[dup].id}): {it.id}")
            else:
                keep.append(i)
        live = [live[i] for i in keep]
        rel = cosine_similarity(vec.transform([query]), vec.transform([it.text for it in live]))[0]
        # 3) score
        scored = []
        for it, r in zip(live, rel):
            s = self.w[0] * r + self.w[1] * it.importance + self.w[2] * self._freshness(it.when) + self.w[3] * it.trust
            scored.append((s, r, it))
        # 4) pinned pehle, phir score ke hisaab se greedy packing
        scored.sort(key=lambda x: (not x[2].pinned, -x[0]))
        chosen, used, per_kind = [], count_tokens(query), {}
        for s, r, it in scored:
            t = count_tokens(it.text)
            cap = self.max_per_kind.get(it.kind)
            if it.pinned:
                ok, why = True, ""
            elif r < self.min_rel:
                ok, why = False, "low relevance"
            elif cap is not None and per_kind.get(it.kind, 0) >= cap:
                ok, why = False, f"kind cap {it.kind}={cap}"
            elif used + t > self.budget:
                ok, why = False, "budget"
            else:
                ok, why = True, ""
            if ok:
                chosen.append(it); used += t; per_kind[it.kind] = per_kind.get(it.kind, 0) + 1
                self.log.append(f"INCLUDED score={s:.2f} rel={r:.2f} {it.id} ({t} tok)")
            else:
                self.log.append(f"EXCLUDED ({why}) score={s:.2f} rel={r:.2f} {it.id}")
        self.used = used
        return self.render(query, chosen)

    @staticmethod
    def render(query, chosen):
        sec = {}
        for it in chosen:
            sec.setdefault(it.kind, []).append(it)
        order = ["memory", "policy", "order", "history"]
        out = []
        for kind in order + [k for k in sec if k not in order]:
            if kind in sec:
                body = "\n".join(f'  <item id="{it.id}" date="{it.when}">{it.text}</item>' for it in sec[kind])
                out.append(f"<{kind}>\n{body}\n</{kind}>")
        out.append(f"<user_question>\n{query}\n</user_question>")
        return "\n".join(out)

# Items ka pool: policies + memory + kuch purani history + ek duplicate
items = [ContextItem(d["id"], "policy", d["text"], importance=0.8,
                     trust=1.0 if d["source"] == "official_policy" else 0.4, when=d["date"],
                     supersedes=["pol-returns-2024"] if d["id"] == "pol-returns-2026" else [])
         for d in POLICY_DOCS]
items += [
    ContextItem("mem-lang", "memory", "User Roman Urdu mein baat karna pasand karta hai.", importance=0.9, trust=0.6, pinned=True),
    ContextItem("mem-city", "memory", "User Multan mein rehta hai.", importance=0.7, trust=0.6, when="2026-09-01"),
    ContextItem("hist-1", "history", "User ne pichle hafte Smart Fit X ke strap ka rang poocha tha.", importance=0.2, when="2026-10-10"),
    ContextItem("pol-shipping-copy", "policy", POLICY_DOCS[2]["text"], importance=0.8, trust=1.0, when="2026-05-01"),
]
items += [ContextItem(o["order_id"], "order", orders_as_text([o]), importance=0.3, when="2026-10-01") for o in ORDERS[:30]]

q = "Main Multan mein hoon, Rs 8000 ka order karun to delivery kitne din mein aur shipping charge kitna?"
# max_per_kind = har category ka hissa (6.1 ka budget allocation idea).
# Routing rule: sawal mein koi order ID nahi → order records ka hissa 0.
builder = ContextBuilder(budget_tokens=600, min_relevance=0.1,
                         max_per_kind={"order": 3 if re.search(r"ORD-\d+", q) else 0, "history": 1})
assembled = builder.build(q, items)
print("\n".join(l for l in builder.log if "INCLUDED" in l or "superseded" in l or "duplicate" in l))
print(f"... aur {sum('EXCLUDED' in l for l in builder.log)} items EXCLUDED (low relevance / kind cap / budget)")
print(f"\nAssembled context ({builder.used} tokens):\n{assembled}")

### Naive "sab kuch daal do" vs Assembled — asli LLM par muqabla

In [ ]:
SYS = "Tum TimeCraft PK ke support assistant ho. Sirf diye gaye context se jawab do, Roman Urdu mein, 2-3 lines."
naive_ctx = "\n".join(it.text for it in items) + f"\n\nSawal: {q}"

r_naive = llm([{"role": "system", "content": SYS}, {"role": "user", "content": naive_ctx}], tag="6.2-naive")
r_asm = llm([{"role": "system", "content": SYS}, {"role": "user", "content": assembled}], tag="6.2-assembled")

for name, r in [("NAIVE", r_naive), ("ASSEMBLED", r_asm)]:
    u = r["usage"]
    print(f"--- {name}: prompt_tokens={u['prompt_tokens']} latency={u['latency']}s cost=${cost_usd(u):.6f}")
    print(wrap(r["text"]), "\n")
print("Expected: 4-6 working days, Rs 250 shipping (order Rs 10,000 se kam hai)")

**Sabaq:** dono shayad sahi jawab dein — magar assembled version **kam tokens, kam latency, kam kharcha** aur kam ghalti ka imkaan (koi ghair mutalliqa order/policy nahi). Scale par (10,000 requests/din) yahi farq bill ko kayi guna kam karta hai.

### 🎤 Interview angle (6.2)
- **Write / Select / Compress / Isolate** framework batayein.
- *"Window mein cheezon ki tarteeb kyun aham hai?"* → (1) stable cheezen pehle = prefix caching, (2) sawal aakhir mein = recency, (3) tags se instruction vs data ka farq = injection se bachao.
- *"Context mein kya jayega, kaun tay karta hai?"* → Code (deterministic builder) — model nahi. Model ko tools de kar khud retrieve bhi karwa sakte hain (agentic retrieval), magar budget/limits harness tay karta hai.

✅ **Outcome:** Is waqt ke faislay ke liye sab se chhota kaam ka context banana.

---
# 6.3 · Compression & Compaction
*"Kam jagah mein zyada."*

## 📖 Theory

### Summarizing History
Lambi conversation ka purana hissa summary mein badalna. **Magar** summary mein yeh cheezen **hamesha** mehfooz rehni chahiye:
1. **Goal** — user asal mein kya chahta hai
2. **Constraints** — shartain ("Saturday ke baad delivery nahi", "budget 15k")
3. **Decisions** — jo tay ho chuka ("exchange approve", "charge waive")
4. **Open tasks** — adhoora kaam
5. **Exact data** — IDs, raqam, tareekhein, pate

Free-form summary ("user ne exchange ki baat ki") aksar **4 aur 5** kho deti hai. Is liye **structured summary** (fixed fields / JSON) behtar hai.

### Compaction
Bare tool outputs ya history ko chhota kar ke wapas dena. Teen darje (halke se bhari):
| Darja | Kya | Nuqsan |
|---|---|---|
| **Tool-result clearing / observation masking** | Purane tool results ki jagah "[cleared — result_id=r7]" | Sab se kam — maloomat bahar mehfooz, zaroorat par dobara |
| **Deterministic compaction** | JSON se faltu fields hatana, sirf top-N rows | Kam, predictable |
| **LLM summarization** | Model se khulasa likhwana | Sab se zyada — model bhool/ghalti kar sakta hai, extra call ka kharcha |

> Claude Code ka `/compact` ya auto-compact isi ki misaal hai: window bharne lage to purani baatcheet ka structured khulasa bana kar naya context shuru.

### Rolling & Hierarchical Summaries
- **Rolling:** ek chalti summary + aakhri K turns verbatim. Jab history had se barhe → sab se purane turns summary mein "fold" ho jate hain.
- **Hierarchical:** chunks ki summaries, phir summaries ki summary (map-reduce). Bohot lambi history/docs ke liye.

### Preserving Exact Data
Numbers, IDs, code — LLM summary mein **sab se pehle** bigarte hain (Rs 26,500 → "taqreeban 26 hazaar"). Inhein **code (regex/parser)** se nikaal kar alag **pinned facts** mein rakhein.

### Reduction vs Information Loss
Jitna chhota, utna khone ka khatra. Is liye compression ko **eval** karein: probe questions jinka jawab pehle se pata ho.

## 💻 Code 1 — Ek lambi support conversation (ground truth ke saath)

In [ ]:
KEY_TURNS = [
    ("user", "Assalam o alaikum, mera order ORD-2041 hai, Diver Pro 200m, Rs 26,500 ka."),
    ("assistant", "Walaikum assalam! Ji ORD-2041 mil gaya. Kya masla hai?"),
    ("user", "Mujhe blue dial aaya hai, maine black order kiya tha. Exchange chahiye."),
    ("user", "Aur please delivery Saturday 25 October ke baad nahi honi chahiye, mujhe gift dena hai."),
    ("assistant", "Samajh gaya. Exchange policy ke mutabiq pehli exchange free hai."),
    ("user", "Mera address bhi badal gaya hai: House 12, Street 4, Gulberg III, Lahore."),
    ("assistant", "Address update kar diya: House 12, Street 4, Gulberg III, Lahore."),
    ("assistant", "Exchange approve ho gaya. Ghalti hamari thi is liye Rs 300 charge bhi waive kar diya hai."),
    ("assistant", "Courier pickup abhi schedule karna baqi hai — main kal subah confirm karunga."),
]
FILLER = [("user", "Aap ki watches ki quality achhi hai, mera bhai bhi lena chahta hai."),
          ("assistant", "Shukriya! Aap ke bhai ke liye Aviator Chrono acha option hai."),
          ("user", "Smart Fit X ki battery kitni chalti hai?"),
          ("assistant", "Taqreeban 7 din normal use par."),
          ("user", "Kya aap ke paas ladies watches bhi hain?"),
          ("assistant", "Abhi nahi, magar agle mahine collection aa raha hai."),
          ("user", "Theek hai, aur wallet combo mein kaunse rang hain?"),
          ("assistant", "Black aur brown dono available hain.")]

rng = random.Random(3)
CONVO = []
for t in KEY_TURNS:
    CONVO.append(t)
    for _ in range(2):
        CONVO.append(rng.choice(FILLER))
convo_text = "\n".join(f"{r}: {c}" for r, c in CONVO)

PROBES = [  # (sawal, expected substrings — sab maujood hon to pass)
    ("Order ID kya hai?", ["2041"]),
    ("Order ki raqam kitni hai?", ["26500"]),
    ("User ka naya address kya hai?", ["house 12", "street 4", "gulberg"]),
    ("Delivery ki aakhri tareekh kya hai?", ["25"]),
    ("Kya exchange charge liya ja raha hai?", ["waive"]),
    ("Kaunsa kaam abhi baqi hai?", ["pickup"]),
]
print(f"{len(CONVO)} turns | {count_tokens(convo_text)} tokens")

## 💻 Code 2 — Teen compression tareeqe + Info-loss Eval
- **A. Full history** (baseline, koi compression nahi)
- **B. Free-form summary** ("3 lines mein summary")
- **C. Structured summary** (JSON: goal / constraints / decisions / open_tasks / facts)
- **D. Structured + pinned exact facts** (regex se nikale IDs, raqam, tareekh, pata)

Phir har context par wahi 6 probe sawal (ek hi call mein JSON jawab) aur score.

In [ ]:
def freeform_summary(text):
    return ask(f"Is conversation ka 3 lines mein summary likho:\n\n{text}", tag="6.3-summ")

STRUCT_PROMPT = """Neeche support conversation hai. Iska STRUCTURED summary JSON mein do, in keys ke saath:
{"goal": str, "constraints": [str], "decisions": [str], "open_tasks": [str], "facts": [str]}
Qawaid: IDs, raqam, tareekhein aur pata BILKUL waise hi likho jaise conversation mein hain. Chitchat chhor do.

CONVERSATION:
"""
def structured_summary(text):
    r = llm(STRUCT_PROMPT + text, json_mode=True, tag="6.3-summ", max_tokens=2000)
    try:
        return json.loads(r["text"])
    except json.JSONDecodeError:
        return {"raw": r["text"]}

def extract_pinned_facts(text):
    """Deterministic — LLM nahi. Exact data ki hifazat."""
    facts = {
        "order_ids": sorted(set(re.findall(r"\bORD-\d+\b", text))),
        "amounts": sorted(set(re.findall(r"Rs\.?\s?[\d,]+", text))),
        "dates": sorted(set(re.findall(r"\b\d{1,2}\s(?:January|February|March|April|May|June|July|August|September|October|November|December)\b", text))),
        "addresses": sorted(set(re.findall(r"House \d+[^.\n]*", text))),
    }
    return {k: v for k, v in facts.items() if v}

def probe_eval(context, label):
    qs = "\n".join(f"{i + 1}. {q}" for i, (q, _) in enumerate(PROBES))
    r = llm(f"CONTEXT:\n{context}\n\nSirf context se jawab do. JSON do: {{\"answers\": [\"...\", ...]}} (har sawal ka ek)\n{qs}",
            json_mode=True, tag=f"6.3-eval-{label}", max_tokens=2000)
    try:
        answers = json.loads(r["text"]).get("answers", [])
    except json.JSONDecodeError:
        answers = []
    score = 0
    for i, (_, exp) in enumerate(PROBES):
        a = (answers[i] if i < len(answers) else "").lower().replace(",", "")
        ok = all(e in a for e in exp)
        score += ok
    return score, answers

summ_b = freeform_summary(convo_text)
summ_c = structured_summary(convo_text)
pinned = extract_pinned_facts(convo_text)
ctx_d = json.dumps({"summary": summ_c, "pinned_exact_facts": pinned}, ensure_ascii=False, indent=1)

variants = {"A full": convo_text, "B freeform": summ_b,
            "C structured": json.dumps(summ_c, ensure_ascii=False, indent=1), "D struct+pinned": ctx_d}
eval_rows = []
for label, ctx in variants.items():
    s, ans = probe_eval(ctx, label)
    eval_rows.append((label, count_tokens(ctx), s))
    print(f"{label:<16} tokens={count_tokens(ctx):<5} score={s}/{len(PROBES)}")

print("\nB (freeform):", wrap(summ_b))
print("\nPinned facts (regex):", pinned)

In [ ]:
labels = [r[0] for r in eval_rows]
fig, ax1 = plt.subplots(figsize=(7, 3.2))
ax1.bar(labels, [r[1] for r in eval_rows], color="#9db4d6"); ax1.set_ylabel("context tokens")
ax2 = ax1.twinx(); ax2.plot(labels, [r[2] / len(PROBES) for r in eval_rows], "o-", color="#c0392b")
ax2.set_ylim(0, 1.05); ax2.set_ylabel("probe accuracy")
ax1.set_title("Compression: tokens (bars) vs information kept (line)"); plt.tight_layout(); plt.show()

**Kya dekhna hai:** freeform summary sab se chhoti hai magar aksar raqam/pata/tareekh kho deti hai. Structured + pinned facts **thore** zyada tokens mein lagbhag full-history jitni maloomat bachata hai. **Yahi production pattern hai.**

## 💻 Code 3 — Rolling Memory (chalti summary + aakhri K turns)

In [ ]:
class RollingMemory:
    """Aakhri `keep_last` turns verbatim; history `max_tokens` se barhe to purane turns summary mein fold."""
    UPDATE_PROMPT = """Tum conversation memory maintain karte ho.
PURANA SUMMARY (JSON): {old}
NAYE TURNS jo summary mein shamil karne hain:
{turns}
Updated summary JSON do (keys: goal, constraints, decisions, open_tasks, facts).
Qawaid: purani zaroori baatein mat hatao; IDs/raqam/pata exact; jo task mukammal ho gaya use open_tasks se nikaal do."""

    def __init__(self, keep_last=6, max_tokens=250):
        self.summary, self.turns, self.pinned = {}, [], {}
        self.keep_last, self.max_tokens, self.folds = keep_last, max_tokens, 0

    def add(self, role, content):
        self.turns.append((role, content))
        for k, v in extract_pinned_facts(content).items():          # exact facts har turn par
            self.pinned[k] = sorted(set(self.pinned.get(k, [])) | set(v))
        if count_tokens(self._turns_text(self.turns)) > self.max_tokens and len(self.turns) > self.keep_last:
            old, self.turns = self.turns[:-self.keep_last], self.turns[-self.keep_last:]
            r = llm(self.UPDATE_PROMPT.format(old=json.dumps(self.summary, ensure_ascii=False),
                                              turns=self._turns_text(old)),
                    json_mode=True, tag="6.3-rolling", max_tokens=2000)
            try:
                self.summary = json.loads(r["text"])
            except json.JSONDecodeError:
                pass
            self.folds += 1

    @staticmethod
    def _turns_text(turns):
        return "\n".join(f"{r}: {c}" for r, c in turns)

    def context(self):
        return (f"<conversation_summary>{json.dumps(self.summary, ensure_ascii=False)}</conversation_summary>\n"
                f"<pinned_facts>{json.dumps(self.pinned, ensure_ascii=False)}</pinned_facts>\n"
                f"<recent_turns>\n{self._turns_text(self.turns)}\n</recent_turns>")

mem = RollingMemory(keep_last=6, max_tokens=250)
sizes, cum = [], 0
for role, c in CONVO:
    mem.add(role, c)
    cum += count_tokens(f"{role}: {c}")
    sizes.append((cum, count_tokens(mem.context())))

print(f"Folds: {mem.folds} | full history: {count_tokens(convo_text)} tok | rolling context: {count_tokens(mem.context())} tok")
s, _ = probe_eval(mem.context(), "rolling")
print(f"Rolling memory probe score: {s}/{len(PROBES)}")
plt.figure(figsize=(6, 3)); plt.plot([a for a, b in sizes], label="full history"); plt.plot([b for a, b in sizes], label="rolling context")
plt.xlabel("turn #"); plt.ylabel("tokens"); plt.legend(); plt.title("Full history linear barhti hai, rolling memory bounded"); plt.tight_layout(); plt.show()

## 💻 Code 4 — Hierarchical (map-reduce) summary
History ko chunks mein todo → har chunk ki summary (map) → summaries ki summary (reduce). Chunks parallel bhi ho sakte hain (L2.7 fan-out/fan-in).

In [ ]:
def hierarchical_summary(turns, chunk_size=10):
    chunks = [turns[i:i + chunk_size] for i in range(0, len(turns), chunk_size)]
    level1 = []
    for i, ch in enumerate(chunks):
        txt = "\n".join(f"{r}: {c}" for r, c in ch)
        level1.append(ask(f"Is hisse ke zaroori facts (IDs, raqam, faislay, shartain, baqi kaam) bullet points mein. Chitchat chhor do.\n\n{txt}",
                          tag="6.3-hier-map"))
    joined = "\n\n".join(f"[chunk {i + 1}]\n{s}" for i, s in enumerate(level1))
    final = ask(f"In chunk summaries ko ek mukammal summary mein jor do. Dohrai hui cheezen ek dafa, exact data mehfooz:\n\n{joined}",
                tag="6.3-hier-reduce")
    return level1, final

lvl1, final_summary = hierarchical_summary(CONVO, chunk_size=10)
print(f"{len(lvl1)} chunk summaries → 1 final ({count_tokens(final_summary)} tokens)\n")
print(wrap(final_summary))

## 💻 Code 5 — Tool output compaction (deterministic, LLM nahi)
Sab se sasta aur bharosemand compression: faltu fields hatao, rows limit karo, aggregate stats do.

In [ ]:
def compact_records(records, keep_fields, max_rows=5, group_by=None):
    out = {"total_rows": len(records), "shown_rows": min(max_rows, len(records)),
           "rows": [{k: r[k] for k in keep_fields if k in r} for r in records[:max_rows]]}
    if group_by:
        counts = {}
        for r in records:
            counts[r[group_by]] = counts.get(r[group_by], 0) + 1
        out[f"count_by_{group_by}"] = counts
    if len(records) > max_rows:
        out["note"] = f"{len(records) - max_rows} rows hidden — poori list ke liye filter ke saath dobara maangein"
    return out

full = json.dumps(ORDERS, ensure_ascii=False)
compact = json.dumps(compact_records(ORDERS, ["order_id", "status", "city"], max_rows=5, group_by="status"), ensure_ascii=False)
print(f"Full tool output: {count_tokens(full):,} tokens → compacted: {count_tokens(compact)} tokens "
      f"({1 - count_tokens(compact) / count_tokens(full):.0%} kam)\n")
print(compact)

### 🎤 Interview angle (6.3)
- *"Chatbot 200 turns ke baad bhool jata hai/kharab hota hai — design batao"* → rolling window (aakhri K verbatim) + structured running summary + pinned exact facts (regex/DB) + long-term memory retrieval (6.4) + compression ka eval.
- *"Summarization mein kya khota hai?"* → exact numbers/IDs, shartain, "kis ne kya kaha" (attribution), negations ("nahi chahiye"), jazbaati lehja. Ilaj: structured schema + deterministic extraction + probe eval.
- *"Compaction vs summarization?"* → compaction structure bachata hai/deterministic ho sakti hai; summarization lossy aur mehngi. Pehle sasta wala (tool-result clearing), phir LLM summary.

✅ **Outcome:** Kaam ki zaroori maloomat khoye baghair context chhota karna.

---
# 6.4 · Memory & State Offloading
*"Context se bahar save karna."*

## 📖 Theory

### Offloading
Maloomat ko context se nikaal kar **bahar** (file, DB, key-value store) rakhna aur **zaroorat par wapas laana**. Context window = RAM (mehngi, chhoti); external store = disk (sasti, bari).

### Scratchpads & External Stores
| Store | Kis liye |
|---|---|
| **Scratchpad** (ek task ke dauran) | Plan, intermediate nateeje, "abhi tak kya pata chala" |
| **Files** (`NOTES.md`, `plan.md`, `progress.json`) | Lambe kaam, multiple context windows (6.8) |
| **DB / KV store** | User preferences, facts (long-term memory) |
| **Vector store** | Purani conversations par semantic search |

### Active vs Stored Context
- **Active** = jo is waqt window mein hai.
- **Stored** = jo bahar mehfooz hai aur ek tool call door hai.
Achha agent sirf ek **pointer/khulasa** active rakhta hai ("notes mein 4 entries hain: preferences, budget...").

### Persisting Plans & Progress
Plan, mukammal steps, faislay aur aham facts **file mein** likho. Agar context reset/compact ho jaye ya crash ho to kaam wahin se shuru (L5.9 Persisting State to Resume).

### Note-Taking Agents
Agent ko `save_note` / `search_notes` **tools** do. Agent khud faisla karta hai kya yaad rakhne layak hai (L5.5 Write → Retrieve → Update → Forget).

### Remembering vs Freeing Space — farq samjhein
| | Memory | Offloading |
|---|---|---|
| **Maqsad** | *Yaad rakhna* — sessions ke darmiyan | *Jagah khali karna* — isi task ke andar |
| **Misaal** | "User leather strap pasand karta hai" | "Ticket #4 ka 3k-token body file mein, context mein sirf 1 line" |
| Dono ka tareeqa ek jaisa (bahar likho, zaroorat par laao), maqsad alag. |

## 💻 Code 1 — File-based Note Store + Note-Taking Agent
Session 1 mein user apni preferences batata hai → agent `save_note` se file mein likhta hai. Phir **context bilkul reset** (naya messages list). Session 2 mein agent `search_notes` se wapas laata hai.

In [ ]:
WORK = Path("ce_workspace"); WORK.mkdir(exist_ok=True)

class NoteStore:
    """JSON file par chhota long-term memory store. (Production: DB + embeddings.)"""
    def __init__(self, path):
        self.path = Path(path)
        self.notes = json.loads(self.path.read_text(encoding="utf-8")) if self.path.exists() else {}

    def _save(self):
        self.path.write_text(json.dumps(self.notes, ensure_ascii=False, indent=1), encoding="utf-8")

    def save_note(self, topic: str, content: str):
        self.notes[topic.strip().lower()] = {"content": content, "updated": time.strftime("%Y-%m-%d %H:%M")}
        self._save()
        return {"saved": topic}

    def search_notes(self, query: str):
        qw = set(re.findall(r"\w+", query.lower()))
        scored = sorted(((len(qw & set(re.findall(r"\w+", (t + " " + n["content"]).lower()))), t, n["content"])
                         for t, n in self.notes.items()), reverse=True)
        hits = [{"topic": t, "content": c} for s, t, c in scored if s > 0][:5]
        return {"results": hits or [{"topic": t, "content": n["content"]} for t, n in list(self.notes.items())[:5]]}

    def list_notes(self):
        return {"topics": list(self.notes)}

notes = NoteStore(WORK / "user_notes.json")
notes.notes = {}; notes._save()                      # demo ke liye saaf shuru

NOTE_TOOLS = [
    tool_schema("save_note", "User ke baare mein koi DURABLE baat yaad rakhne ke liye (preference, budget, city). Har topic ek dafa.",
                {"topic": {"type": "string", "description": "chhota topic, jaise 'budget' ya 'strap_preference'"},
                 "content": {"type": "string"}}, ["topic", "content"]),
    tool_schema("search_notes", "User ke baare mein pehle se save ki gayi baatein dhoondo. Recommendation se PEHLE zaroor chalao.",
                {"query": {"type": "string"}}, ["query"]),
    tool_schema("list_notes", "Sab saved topics ki list."),
]
NOTE_REG = {"save_note": notes.save_note, "search_notes": notes.search_notes, "list_notes": notes.list_notes}
CATALOG = "\n".join(f"- {p['name']} (Rs {p['price']:,}, warranty {p['warranty_months']} mahine)" for p in PRODUCTS)
NOTE_SYS = (f"Tum TimeCraft PK ke shopping assistant ho. Catalog:\n{CATALOG}\n"
            "User ki durable preferences save_note se yaad rakho. Recommendation se pehle search_notes chalao. Roman Urdu, mukhtasar.")

# ---- Session 1
s1 = [{"role": "system", "content": NOTE_SYS},
      {"role": "user", "content": "Mera naam Bilal hai, Karachi se. Mujhe sirf leather strap pasand hai aur budget 15,000 tak hai. Abhi kuch nahi lena, bas yaad rakhna."}]
ans1, s1, _ = run_tool_loop(s1, NOTE_TOOLS, NOTE_REG, tag="6.4-notes")
print("Session 1 jawab:", wrap(ans1))
print("📁 File mein:", json.dumps(notes.notes, ensure_ascii=False, indent=1))

In [ ]:
# ---- CONTEXT RESET: naya messages list — pichli baatcheet ka ek lafz bhi window mein nahi
q2 = "Assalam o alaikum, mere liye koi watch suggest karo."

# (A) Memory tools ke baghair
plain = ask(q2, system=f"Tum TimeCraft PK ke shopping assistant ho. Catalog:\n{CATALOG}\nRoman Urdu, mukhtasar.", tag="6.4-nomem")

# (B) Note store ke saath (fresh context, magar file mein memory)
notes2 = NoteStore(WORK / "user_notes.json")         # disk se dobara load — "naya process"
NOTE_REG2 = {"save_note": notes2.save_note, "search_notes": notes2.search_notes, "list_notes": notes2.list_notes}
s2 = [{"role": "system", "content": NOTE_SYS}, {"role": "user", "content": q2}]
ans2, s2, steps = run_tool_loop(s2, NOTE_TOOLS, NOTE_REG2, tag="6.4-notes")

print("(A) bina memory :", wrap(plain), "\n")
print("(B) note store  :", wrap(ans2))
print("\nExpected: Classic Leather (Rs 9,500) — leather + budget ke andar")

## 💻 Code 2 — Task State File (plan + progress offloading)
Lambe kaam mein state ko file mein rakhna, aur context mein sirf **chhota view** dena.

In [ ]:
class TaskState:
    """Durable task state. Context mein sirf compact_view() jata hai; poora record file mein."""
    def __init__(self, path, goal, steps):
        self.path = Path(path)
        self.data = {"goal": goal, "steps": [{"id": i + 1, "desc": s, "status": "pending", "result": None} for i, s in enumerate(steps)],
                     "decisions": [], "facts": {}}
        self.save()

    def save(self):
        self.path.write_text(json.dumps(self.data, ensure_ascii=False, indent=1), encoding="utf-8")

    def complete(self, step_id, result):
        for s in self.data["steps"]:
            if s["id"] == step_id:
                s["status"], s["result"] = "done", result
        self.save()

    def compact_view(self):
        done = [s for s in self.data["steps"] if s["status"] == "done"]
        nxt = next((s for s in self.data["steps"] if s["status"] == "pending"), None)
        return (f"GOAL: {self.data['goal']}\nDONE: {len(done)}/{len(self.data['steps'])} "
                f"(last: {done[-1]['desc'] + ' → ' + str(done[-1]['result'])[:60] if done else '-'})\n"
                f"NEXT: {nxt['desc'] if nxt else 'ALL DONE'}\n(full details: {self.path.name})")

ts = TaskState(WORK / "task_state.json", "Har shehar ke DELAYED orders ki report",
               [f"{c} ke delayed orders gino" for c in CITIES] + ["report likho"])
for i, c in enumerate(CITIES, 1):
    ts.complete(i, sum(1 for o in ORDERS if o["city"] == c and o["status"] == "DELAYED"))
print("Active context (model ko yeh milta hai):\n" + ts.compact_view())
print(f"\nStored: {count_tokens(ts.path.read_text(encoding='utf-8'))} tokens file mein | Active: {count_tokens(ts.compact_view())} tokens")

### 🎤 Interview angle (6.4)
- *"Memory, offloading aur RAG mein farq?"* → RAG = bahar ki **knowledge** (docs). Memory = **user/conversation** ki durable baatein. Offloading = isi **task** ka working data bahar rakh kar jagah khali karna.
- *"Agent ko kya yaad rakhna chahiye?"* → durable facts/preferences/faislay — har cheez nahi. Write ke saath **update aur forget** bhi (privacy! PII ko memory mein na rakhein — L8.6).
- *"Agent crash ho gaya 30 steps baad"* → state file/checkpoint se resume (LangGraph checkpointing L5.7 isi ka framework version hai).

✅ **Outcome:** Systems jinhein poori history context mein rakhne ki zaroorat na ho.

---
# 6.5 · Long-Context Strategies
*"Bari window kaafi nahi."*

## 📖 Theory

### Why Big Windows Don't Solve It
128k ya 1M window ka matlab sirf yeh hai ke itna text **ghus sakta** hai — yeh nahi ke model har token par barabar dhyan dega. Masail:
- **Lost in the Middle** (Liu et al., 2023): maloomat shuru ya aakhir mein ho to model achha istemal karta hai; **darmiyan** mein ho to accuracy girti hai (U-shaped curve).
- **Context rot** + distractors (6.1)
- Har request par **poore** context ka bill aur prefill latency

### Managing Lost-in-the-Middle
1. **Aham cheez shuru ya aakhir** mein rakho (sawal aakhir mein, hidayat shuru mein)
2. **Selective retrieval** — poora doc nahi, sirf mutalliqa chunks
3. **Reorder** — retrieve kiye chunks mein sab se relevant ko **pehle aur aakhir** mein rakho, kam relevant beech mein ("sandwich"/LongContextReorder pattern)
4. **Restructure** — headings, IDs, tags taake model "navigate" kar sake

### Stuffing vs Retrieval vs Summarization
| Strategy | Kab | Faida | Nuqsan |
|---|---|---|---|
| **Stuffing** (sab daal do) | Doc chhota (window ka ≲ 20-30%), poore doc par reasoning chahiye | Kuch miss nahi hota, aasan | Mehnga, slow, lost-in-middle |
| **Retrieval** (top-k chunks) | Specific factual sawal, bara corpus | Sasta, tez, focused | Retrieval miss = jawab ghalat (L4.8) |
| **Summarization / Map-Reduce** | "Poore doc ka khulasa", themes, global sawal | Poore doc ka nazara | Detail kho jati hai, kayi calls |
| **Hierarchical / Selective loading** | Bohot bare docs (books, codebases) | Pehle outline/TOC, phir zaroorat ke section | Zyada orchestration |

### More vs Better Context
Sawal ki qism faisla karti hai: **"X ka code kya hai?"** → retrieval. **"Poori policy mein kya contradictions hain?"** → stuffing ya map-reduce. **"Saare tickets ka trend?"** → map-reduce / code se aggregation.

## 💻 Code — Ek lamba "Operations Manual" aur 4 strategies ka naap tol
Manual mein 24 sections hain; jawab **darmiyan** wale sections mein chhupaye gaye hain, aur **distractors** bhi hain (doosre warehouse ka code).

In [ ]:
TOPICS = ["Opening checklist", "Packaging standards", "COD cash handling", "Courier handover", "Inventory audit",
          "Warehouse safety", "Karachi warehouse", "Customer calls", "WhatsApp etiquette", "Strap replacement",
          "Battery replacement", "Returns inspection", "Gulberg warehouse", "Eid season operations", "Gift wrapping",
          "Photography for listings", "Social media replies", "Fraud orders", "Refund processing", "Exchange flow",
          "Warranty claims", "Supplier receiving", "Closing checklist", "Escalation matrix"]
rng = random.Random(11)
SECTIONS = []
for i, t in enumerate(TOPICS):
    sents = [f"Section {i + 1}: {t}.",
             f"Is process ka zimmedar shift supervisor hai aur har {rng.randint(2, 9)} ghante baad log update hota hai.",
             f"Reference code {t[:3].upper()}-{rng.randint(100, 999)} system mein darj karein.",
             f"Agar masla ho to {rng.choice(['ops lead', 'store manager', 'finance desk'])} ko {rng.randint(10, 59)} minute ke andar batayein.",
             f"Is kaam ke liye form OP-{rng.randint(10, 99)} bharna lazmi hai aur copy {rng.choice(['email', 'drive', 'register'])} mein rakhein.",
             "Har qadam par customer ki privacy ka khayal rakhein aur ghair zaroori maloomat share na karein."]
    if t == "Karachi warehouse":
        sents.insert(2, "Karachi warehouse ka loading dock access code D-1290 hai (sirf Karachi staff).")
    if t == "Gulberg warehouse":
        sents.insert(3, "Gulberg (Lahore) warehouse ka loading dock access code D-7734 hai.")
    if t == "Eid season operations":
        sents.insert(2, "Eid season mein courier handover ka cut-off time 3:30 PM hai; is ke baad wale orders agle din jayenge.")
    if t == "Returns inspection":
        sents.insert(4, "Damaged box wale return ki inspection 48 ghante ke andar mukammal honi chahiye.")
    SECTIONS.append(" ".join(sents))
MANUAL = "\n\n".join(SECTIONS)

LC_QUESTIONS = [("Gulberg warehouse ka loading dock code kya hai?", "d-7734"),
                ("Eid season mein courier cut-off time kya hai?", "3:30"),
                ("Damaged box wale return ki inspection kitne ghante mein honi chahiye?", "48")]
print(f"Manual: {len(SECTIONS)} sections, {count_tokens(MANUAL):,} tokens")

In [ ]:
_sec_vec = TfidfVectorizer(ngram_range=(1, 2)).fit(SECTIONS)
_sec_mat = _sec_vec.transform(SECTIONS)

def retrieve(q, k=3):
    sims = cosine_similarity(_sec_vec.transform([q]), _sec_mat)[0]
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

def reorder_for_edges(ranked):
    """Best chunks ko kinaron (start/end) par: [1,3,5,...,6,4,2] — lost-in-the-middle se bachao."""
    front, back = [], []
    for j, item in enumerate(ranked):
        (front if j % 2 == 0 else back).append(item)
    return front + back[::-1]

ANSWER_RULE = "Sirf diye gaye text se jawab do, ek line. Agar na mile to 'NOT FOUND'."

def strat_stuff(q):
    return f"{MANUAL}\n\nSawal: {q}"

def strat_retrieve(q, k=3):
    ctx = "\n\n".join(SECTIONS[i] for i, _ in retrieve(q, k))
    return f"{ctx}\n\nSawal: {q}"

def strat_reorder(q, k=6):
    ctx = "\n\n".join(SECTIONS[i] for i, _ in reorder_for_edges(retrieve(q, k)))
    return f"{ctx}\n\nSawal: {q}"

# Map-reduce: question-agnostic khulasa EK dafa banta hai, phir har sawal par istemal
MR_SUMMARIES = None
def strat_mapreduce(q, group=6):
    global MR_SUMMARIES
    if MR_SUMMARIES is None:
        MR_SUMMARIES = [ask("In sections ka mukhtasar khulasa (2-3 lines har section):\n\n" + "\n\n".join(SECTIONS[i:i + group]),
                            tag="6.5-mapreduce-map") for i in range(0, len(SECTIONS), group)]
    return "\n\n".join(MR_SUMMARIES) + f"\n\nSawal: {q}"

STRATS = {"stuffing": strat_stuff, "retrieval k=3": strat_retrieve, "retrieve+reorder k=6": strat_reorder, "map-reduce summary": strat_mapreduce}
lc_rows = []
for name, fn in STRATS.items():
    ok, toks, lat = 0, [], []
    for q, exp in LC_QUESTIONS:
        r = llm([{"role": "system", "content": ANSWER_RULE}, {"role": "user", "content": fn(q)}], tag=f"6.5-{name}")
        hit = exp in r["text"].lower()
        ok += hit; toks.append(r["usage"]["prompt_tokens"]); lat.append(r["usage"]["latency"])
    lc_rows.append((name, ok, int(np.mean(toks)), round(float(np.mean(lat)), 2)))

print(f"{'strategy':<24}{'correct':>8}{'avg tokens':>12}{'avg latency':>13}")
for n, ok, t, l in lc_rows:
    print(f"{n:<24}{ok}/{len(LC_QUESTIONS):<6}{t:>12,}{l:>12}s")

**Aam nateeja:** retrieval wale strategies **bohot kam tokens** mein utne hi sahi jawab dete hain jitna stuffing; map-reduce summary **exact codes/times kho deti hai** (khulasa question-agnostic tha) — yahi uska trade-off hai. Stuffing yahan isliye kaam kar gaya ke doc chhota hai; 100k+ tokens par fark nazar aata hai.

## 💻 Decision helper — strategy kaise chunein

In [ ]:
def choose_strategy(doc_tokens, window, question_type, budget_frac=0.3):
    """question_type: 'specific_fact' | 'global_summary' | 'cross_reference' | 'aggregation'"""
    fits = doc_tokens <= window * budget_frac
    if question_type == "aggregation":
        return "Code/tool se compute karo (SQL, pandas) — LLM se ginti mat karwao"
    if question_type == "specific_fact":
        return "Retrieval (hybrid + rerank) → top-k reorder" if not fits else "Retrieval (sasta) — stuffing bhi chal jayega"
    if question_type == "global_summary":
        return "Stuffing" if fits else "Map-reduce / hierarchical summary"
    if question_type == "cross_reference":
        return "Stuffing (poora doc chahiye)" if fits else "Hierarchical: TOC/outline → zaroori sections load → reasoning"
    return "Pehle sawal ki qism tay karo"

for args in [(3_000, 128_000, "specific_fact"), (400_000, 128_000, "global_summary"),
             (20_000, 128_000, "cross_reference"), (50_000, 128_000, "aggregation")]:
    print(args, "→", choose_strategy(*args))

### 🎤 Interview angle (6.5)
- *"Gemini/GPT ke paas 1M context hai, RAG khatam?"* → Nahi: cost per request, latency, lost-in-middle, context rot, access control (har user har doc nahi dekh sakta — L8.5), freshness. Long context aur RAG **mil kar** chalte hain.
- *"Lost in the middle kya hai aur ilaj?"* → U-shaped attention; ilaj: retrieval, reranking, aham cheez kinaron par, sawal aakhir mein, chhota context.

✅ **Outcome:** Bohot lambe documents, conversations aur agent runs ke liye strategy chunna.

---
# 6.6 · Tool Context Management
*"Tools bhi jagah lete hain."*

## 📖 Theory

### Tool Definitions Compete for Space
Har tool ka naam + description + JSON schema **har request** ke saath jata hai. 50 tools = aksar hazaron tokens — sawal poochne se pehle hi! Upar se **context confusion**: milte julte tools mein model ghalat chunta hai (L2.6 Too Many Tools). Aik MCP server jor diya to achanak 30 tools aur aa gaye.

### Dynamic Tool Exposure
Sirf is waqt ke mutalliqa tools dikhao:
- **Rule/router based:** user intent → tool group (orders, inventory, hr…)
- **Tool retrieval ("tool RAG"):** tool descriptions par search, top-k tools
- **Tool-search meta tool:** model ko ek `find_tools(query)` tool do jo zaroorat par naye tools "load" kare
- **Agent ke stage ke hisaab se:** planning phase mein read-only tools, execution phase mein write tools

### Filtering Large Outputs
API/DB/search ke nateeje aksar bohot bare hote hain. Model ko dene se **pehle** filter/compact/summarize karo (6.3 Code 5).

### Store Full, Re-retrieve Details (Result Handles)
Poora nateeja bahar save karo, context mein sirf **khulasa + handle** (`result_id`). Model ko detail chahiye to `query_result(result_id, filter, sort, limit)` se mangwaye. Yeh "lazy loading" hai — file system ki tarah.

### Observation Build-up
Lambe agent loop mein har tool result context mein jama hota jata hai. 30 steps baad 90% window purane observations. Ilaj: **observation masking** — aakhri N tool results rakho, purane ki jagah ek line placeholder (maloomat handle ke zariye wapas mil sakti hai). Yeh sab se sasti compaction hai aur research mein aksar LLM-summary jitni hi achhi nikli hai.

## 💻 Code 1 — 40 tools ka bojh aur Dynamic Tool Exposure

In [ ]:
DOMAINS = {
    "orders": ["get_order_status", "cancel_order", "update_order_address", "list_recent_orders", "search_orders"],
    "inventory": ["check_stock", "reserve_stock", "list_low_stock", "update_stock_count", "get_sku_details"],
    "shipping": ["track_shipment", "schedule_pickup", "get_courier_rates", "print_label", "report_lost_parcel"],
    "returns": ["create_return", "approve_refund", "get_return_status", "create_exchange", "check_warranty"],
    "marketing": ["create_discount_code", "list_campaigns", "send_broadcast", "get_campaign_stats", "pause_campaign"],
    "finance": ["get_daily_sales", "reconcile_cod", "get_invoice", "list_payouts", "export_ledger"],
    "hr": ["get_staff_schedule", "approve_leave", "list_staff", "log_attendance", "get_payroll_summary"],
    "analytics": ["get_top_products", "get_city_breakdown", "get_conversion_rate", "get_repeat_customers", "get_refund_rate"],
}
DESC = {"get": "maloomat lata hai", "list": "list deta hai", "create": "naya banata hai", "update": "update karta hai",
        "cancel": "cancel karta hai", "check": "check karta hai", "approve": "approve karta hai", "track": "track karta hai"}
ALL_TOOLS = []
for dom, names in DOMAINS.items():
    for n in names:
        verb = n.split("_")[0]
        ALL_TOOLS.append(tool_schema(
            n, f"[{dom}] {n.replace('_', ' ')} — {DESC.get(verb, 'kaam karta hai')}. {dom.capitalize()} domain ka tool.",
            {"id": {"type": "string", "description": f"{dom} se mutalliq ID (order_id, sku, staff_id...)"},
             "date_from": {"type": "string", "description": "YYYY-MM-DD (optional)"},
             "limit": {"type": "integer", "description": "zyada se zyada rows"}}, ["id"]))

all_tools_tokens = count_tokens(json.dumps(ALL_TOOLS))
print(f"{len(ALL_TOOLS)} tools → {all_tools_tokens:,} tokens (sirf definitions, har request par!)")

_tool_docs = [t["function"]["name"].replace("_", " ") + " " + t["function"]["description"] for t in ALL_TOOLS]
_tool_vec = TfidfVectorizer().fit(_tool_docs)
_tool_mat = _tool_vec.transform(_tool_docs)

def select_tools(query, k=5):
    sims = cosine_similarity(_tool_vec.transform([query]), _tool_mat)[0]
    return [ALL_TOOLS[i] for i in np.argsort(-sims)[:k]]

TOOL_SYS = "Tum TimeCraft PK ke ops assistant ho. Sahi tool chuno. Agar ID na ho to jo user ne diya wahi do."
tq = "Order ORD-1042 ka shipment kahan pohancha? Track karo."
for label, tools in [("ALL 40 tools", ALL_TOOLS), ("TOP-5 selected", select_tools(tq, 5))]:
    r = llm([{"role": "system", "content": TOOL_SYS}, {"role": "user", "content": tq}], tools=tools, tag=f"6.6-{label}")
    tcs = getattr(r["message"], "tool_calls", None) or []
    picked = [tc.function.name for tc in tcs]
    print(f"{label:<16} prompt_tokens={r['usage']['prompt_tokens']:<6} picked={picked}  latency={r['usage']['latency']}s")
print("Selected:", [t["function"]["name"] for t in select_tools(tq, 5)])

## 💻 Code 2 — Result Handles: "store full, return summary, re-retrieve on demand"
Sawal: *"Lahore ke DELAYED orders mein se 3 sab se mehngay orders ke IDs aur unki kul raqam?"*
- **Naive agent:** `get_all_orders()` → poora 60-order JSON context mein
- **Managed agent:** `search_orders()` → result **store** hota hai, context mein sirf khulasa + `result_id`; `query_result()` se sort/filter

In [ ]:
class ResultStore:
    def __init__(self):
        self.store, self.n = {}, 0

    def put(self, rows):
        self.n += 1
        rid = f"r{self.n}"
        self.store[rid] = rows
        return rid

RSTORE = ResultStore()

def get_all_orders():
    return ORDERS                                         # naive: sab kuch wapas

def search_orders(city: str = None, status: str = None):
    rows = [o for o in ORDERS if (not city or o["city"].lower() == city.lower())
            and (not status or o["status"].upper() == status.upper())]
    rid = RSTORE.put(rows)
    return {"result_id": rid, "count": len(rows),
            "preview": [{"order_id": r["order_id"], "amount_pkr": r["amount_pkr"]} for r in rows[:3]],
            "hint": "Poori/sorted rows ke liye query_result(result_id, sort_by, descending, limit, fields) chalao."}

def query_result(result_id: str, sort_by: str = None, descending: bool = True, limit: int = 5, fields: list = None):
    rows = RSTORE.store.get(result_id)
    if rows is None:
        return {"error": f"result_id {result_id} nahi mila"}
    if sort_by:
        rows = sorted(rows, key=lambda r: r.get(sort_by, 0), reverse=descending)
    limit = max(1, min(int(limit or 5), 20))              # harness cap — model 10,000 rows na mangwa sake
    fields = fields or ["order_id", "amount_pkr", "status", "city"]
    return {"rows": [{f: r.get(f) for f in fields} for r in rows[:limit]], "total": len(rows)}

lah = sorted([o for o in ORDERS if o["city"] == "Lahore" and o["status"] == "DELAYED"], key=lambda o: -o["amount_pkr"])[:3]
TRUTH_IDS, TRUTH_SUM = {o["order_id"] for o in lah}, sum(o["amount_pkr"] for o in lah)
print("Ground truth:", TRUTH_IDS, TRUTH_SUM)

Q66 = "Lahore ke DELAYED orders mein se 3 sab se mehngay orders ke IDs aur unki kul raqam (PKR) batao."
SYS66 = "Tum ops assistant ho. Tools istemal karo. Aakhir mein 'IDS: ...' aur 'TOTAL: <number>' likho."

naive_tools = [tool_schema("get_all_orders", "Store ke tamam orders ki poori list.")]
ans_n, msgs_n, _ = run_tool_loop([{"role": "system", "content": SYS66}, {"role": "user", "content": Q66}],
                                 naive_tools, {"get_all_orders": get_all_orders}, tag="6.6-naive-handles")
managed_tools = [
    tool_schema("search_orders", "City/status se orders filter karo. Result store hota hai; result_id wapas milta hai.",
                {"city": {"type": "string"}, "status": {"type": "string", "enum": STATUSES}}),
    tool_schema("query_result", "Stored result ko sort/limit kar ke rows lao.",
                {"result_id": {"type": "string"}, "sort_by": {"type": "string", "enum": ["amount_pkr", "order_id"]},
                 "descending": {"type": "boolean"}, "limit": {"type": "integer"},
                 "fields": {"type": "array", "items": {"type": "string"}}}, ["result_id"]),
]
ans_m, msgs_m, _ = run_tool_loop([{"role": "system", "content": SYS66}, {"role": "user", "content": Q66}],
                                 managed_tools, {"search_orders": search_orders, "query_result": query_result}, tag="6.6-managed-handles")

def grade66(text):
    ids = set(re.findall(r"ORD-\d+", text))
    m = re.search(r"TOTAL:\s*(?:Rs\.?\s*)?([\d,]+)", text)
    total = int(m.group(1).replace(",", "")) if m else None
    return ids == TRUTH_IDS, total == TRUTH_SUM

for name, ans, tag in [("NAIVE", ans_n, "6.6-naive-handles"), ("MANAGED", ans_m, "6.6-managed-handles")]:
    print(f"\n{name}: ids_ok,total_ok={grade66(ans)} | {usage_summary(tag)}")
    print(wrap(ans)[:300])

## 💻 Code 3 — Observation Masking (purane tool results saaf karna)
Yeh function `run_tool_loop` ke `on_step` hook mein lagta hai — capstone mein isi ko istemal karenge.

In [ ]:
def mask_old_tool_results(messages, keep_last=2, min_chars=300):
    """Aakhri `keep_last` tool results ke ilawa sab bare tool results ko ek-line placeholder se badlo.
    Message structure (assistant tool_calls ↔ tool) waise hi rehta hai — sirf content chhota hota hai."""
    tool_idx = [i for i, m in enumerate(messages) if m.get("role") == "tool"]
    masked = 0
    for i in tool_idx[:-keep_last] if keep_last else tool_idx:
        c = messages[i]["content"]
        if len(c) > min_chars and not c.startswith("[masked"):
            messages[i]["content"] = f"[masked: purana tool result ({len(c)} chars) — zaroorat ho to tool dobara chalao]"
            masked += 1
    return masked

# Demo: 6 bare fake observations
demo = [{"role": "system", "content": "sys"}, {"role": "user", "content": "task"}]
for i in range(6):
    demo.append({"role": "assistant", "content": None, "tool_calls": [{"id": f"c{i}", "type": "function", "function": {"name": "get_all_orders", "arguments": "{}"}}]})
    demo.append({"role": "tool", "tool_call_id": f"c{i}", "content": json.dumps(ORDERS[:20])})
before = count_msg_tokens(demo)
n = mask_old_tool_results(demo, keep_last=1)
print(f"Masked {n} results: {before:,} → {count_msg_tokens(demo):,} tokens")

### 🎤 Interview angle (6.6)
- *"Agent ke paas 80 tools hain, accuracy gir rahi hai aur kharcha zyada"* → token count naapo; tools ko groups/namespaces mein; router ya tool-retrieval se top-k; overlapping tools merge; descriptions saaf (L2.6); eval se pehle/baad tool-selection accuracy naapo (L7.9).
- *"Tool ne 50k tokens ka JSON diya"* → harness level par filter/paginate, result handle + summary, model ko query tool do; kabhi raw dump context mein nahi.
- *"Observation masking vs summarization?"* → masking sasti, deterministic, maloomat handle se wapas mil sakti hai; summarization extra LLM call + lossy.

✅ **Outcome:** Tool-heavy agents ko coherent rakhna.

---
# 6.7 · Context Caching & Efficiency
*"Static context par paisa bachana."*

## 📖 Theory

### Cost of Resending Static Context
API **stateless** hai (L2.1) — har request par poora system prompt, tool definitions, docs dobara jate hain. 3,000 token ka system prompt × 10,000 requests/din = 3 crore input tokens/din sirf dohrai hui cheez ke!

### Prompt (Prefix) Caching kaise kaam karti hai
L1.5 ka **KV cache** yaad karein: prefill mein har token ke Key/Value vectors bante hain. Agar do requests ka **shuruati hissa (prefix) bilkul ek jaisa** ho, provider pichli request ke K/V dobara istemal kar leta hai:
- **Latency kam** (prefill skip) aur aksar **cached tokens sasta** (provider ke hisaab se discount)
- Sirf **exact prefix match** — beech mein ek character badla to us ke baad sab cache miss

### Structuring for Cache Hits
```
✅ GOOD                              ❌ BAD
[system: stable rules]               [system: "Time: 17:03:22, req_id=8f2a..."]  ← har dafa naya!
[tool definitions — fixed order]     [system: stable rules]
[static docs / few-shot examples]    [tools — random order]
[conversation history (append-only)] [docs]
[latest user message]                [user message]
```
**Cache todne wali aam ghaltiyan:** prompt ke shuru mein timestamp/request-id/user-name, tools ki tarteeb badalna, JSON keys ki random order, history ko beech se edit karna (summary ko beech mein dalna), har request par alag few-shot examples.

> **Trade-off:** 6.3 ki compaction/masking history ko *badalti* hai → us point ke baad cache miss. Is liye compaction **kabhi kabhi, batch mein** karo, har step par nahi; aur stable hissa hamesha upar.

### Stable vs Changing State
Badalne wali cheezen (current time, user state, task progress) **aakhri message** mein dalo, system prompt mein nahi.

### Measuring Cache Utilization
Andhe optimization nahi — naapo: `usage.prompt_tokens_details.cached_tokens`, hit rate = cached / prompt tokens, latency, cost. Groq par caching automatic hai (koi code change nahi) magar **sirf supported models** par — model aur account ke hisaab se `cached_tokens` 0 bhi aa sakta hai. Yahi wajah hai ke hum *naap* rahe hain, farz nahi kar rahe.

## 💻 Code 1 — Good vs Bad prompt structure (asli `cached_tokens` naap)

In [ ]:
import uuid
STATIC_KB = ("Tum TimeCraft PK ke senior support assistant ho. Qawaid: sirf knowledge base se jawab, Roman Urdu, 2 lines.\n\n"
             "<policies>\n" + "\n".join(f"[{d['id']}] {d['text']}" for d in POLICY_DOCS) + "\n</policies>\n"
             "<catalog>\n" + CATALOG + "\n</catalog>\n<ops_manual>\n" + MANUAL + "\n</ops_manual>")
print("Static prefix:", count_tokens(STATIC_KB), "tokens")

CACHE_QS = ["Diver Pro ki warranty kitni hai?", "Multan delivery kitne din?", "Exchange ka charge?",
            "Eid par courier cut-off?", "Refund kitne din mein aata hai?"]

def run_cache_test(label, build):
    rows = []
    for q in CACHE_QS:
        r = llm(build(q), tag=f"6.7-{label}", max_tokens=800)
        u = r["usage"]
        rows.append((q[:28], u["prompt_tokens"], u["cached_tokens"], u["latency"]))
        time.sleep(0.5)
    return rows

good = lambda q: [{"role": "system", "content": STATIC_KB},
                  {"role": "user", "content": f"(time: {time.strftime('%H:%M:%S')}) {q}"}]        # dynamic cheez AAKHIR mein
bad = lambda q: [{"role": "system", "content": f"Request {uuid.uuid4()} at {time.time()}\n" + STATIC_KB},  # dynamic cheez SHURU mein
                 {"role": "user", "content": q}]

cache_tbl = {"GOOD": run_cache_test("good", good), "BAD": run_cache_test("bad", bad)}
for label, rows in cache_tbl.items():
    print(f"\n{label} structure")
    print(f"{'question':<30}{'prompt':>8}{'cached':>8}{'latency':>9}")
    for q, p, c, l in rows:
        print(f"{q:<30}{p:>8}{c:>8}{l:>8.2f}s")
    tp, tc = sum(r[1] for r in rows), sum(r[2] for r in rows)
    print(f"cache hit rate: {tc / tp:.0%}")
if all(r[2] == 0 for rows in cache_tbl.values() for r in rows):
    print("\nℹ️ cached_tokens sab 0 — is model/account par Groq prompt caching abhi active nahi. "
          "Supported models ke liye console.groq.com/docs/prompt-caching dekhein. Structure ka usool phir bhi wahi hai "
          "(OpenAI, Anthropic, Gemini, DeepSeek sab exact-prefix caching karte hain).")

## 💻 Code 2 — Cost calculator (interview mein aksar aata hai)

In [ ]:
def monthly_cost(req_per_day, static_prefix, dynamic_in, out_tokens, price_in, price_out,
                 cache_hit_rate=0.0, cached_discount=0.5, days=30):
    """USD/month. price_* = USD per 1M tokens."""
    per_req_in = static_prefix + dynamic_in
    cached = static_prefix * cache_hit_rate
    in_cost = ((per_req_in - cached) * price_in + cached * price_in * (1 - cached_discount)) / 1e6
    out_cost = out_tokens * price_out / 1e6
    return round((in_cost + out_cost) * req_per_day * days, 2)

scenario = dict(req_per_day=10_000, static_prefix=4_000, dynamic_in=600, out_tokens=300, price_in=0.15, price_out=0.60)
base = monthly_cost(**scenario)
print(f"No caching             : ${base}")
for hit in (0.5, 0.9):
    print(f"Cache hit {hit:.0%} (50% off): ${monthly_cost(**scenario, cache_hit_rate=hit)}")
print(f"Prefix 4000→1500 (context trim) + 90% hit: ${monthly_cost(**{**scenario, 'static_prefix': 1500}, cache_hit_rate=0.9)}")

## 💻 Code 3 — Client-side exact response cache
Provider ki prefix caching ke ilawa, **bilkul wahi request** (FAQ, temperature 0) ka jawab apni taraf save kar lo. (Milte julte sawalon ki **semantic cache** L9.5 mein.)

In [ ]:
class ResponseCache:
    def __init__(self):
        self.store, self.hits, self.misses = {}, 0, 0

    def key(self, model, messages):
        return hashlib.sha256(json.dumps({"m": model, "msgs": messages}, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

    def ask(self, messages, model=None, **kw):
        k = self.key(model or FAST_MODEL, messages)
        if k in self.store:
            self.hits += 1
            return self.store[k], 0.0
        self.misses += 1
        r = llm(messages, model=model, temperature=0, tag="6.7-respcache", **kw)
        self.store[k] = r["text"]
        return r["text"], r["usage"]["latency"]

rc = ResponseCache()
faq = [{"role": "system", "content": STATIC_KB}, {"role": "user", "content": "Shipping free kab hoti hai?"}]
for i in range(3):
    txt, lat = rc.ask(faq)
    print(f"call {i + 1}: latency={lat:.2f}s  | {txt[:70]}")
print(f"hits={rc.hits} misses={rc.misses}")

### 🎤 Interview angle (6.7)
- *"Prompt caching ke liye prompt kaise design karoge?"* → stable → dynamic tarteeb, tools fixed order, history append-only, timestamps aakhir mein, cache hit rate monitor.
- *"Caching aur compaction mein takrao?"* → compaction prefix badalti hai → cache reset. Threshold par batch mein compact karo, stable hissa chhuo mat.
- Cost calculation ka formula zubani yaad rakhein: `requests × (fresh_in × p_in + cached × p_in × (1-d) + out × p_out)`.

✅ **Outcome:** Large-context apps ko quality khoye baghair efficient banana.

---
# 6.8 · Harness Engineering
*"Model ke gird ka poora dhancha."*

## 📖 Theory

### Agent Harness
Model ke ird gird **sab kuch** jo use agent banata hai:
```
┌──────────────────────────── HARNESS ─────────────────────────────┐
│  loop (step/time/token limits)     permissions & approval gates  │
│  system prompt + AGENTS.md         tools + sandbox/environment   │
│  context manager (select/compress/mask)   state & durable files  │
│  verification (tests/linters)      sub-agents   logging/tracing  │
│                      ┌───────────────┐                           │
│                      │     MODEL     │  ← sirf "agla qadam kya"  │
│                      └───────────────┘                           │
└──────────────────────────────────────────────────────────────────┘
```
Claude Code, Codex, Cursor agent — sab ek **model + harness** hain. Aap jo CLAUDE.md, hooks, skills, sub-agents istemal karte hain (L0.6) — woh sab harness ke parts hain.

### Harness vs Model Problems
Wahi model ek harness mein kamyab aur doosre mein nakam. Agent fail ho to pehle poochein: tools saaf the? context mein sahi cheez thi? loop ne error wapas diya? limits theek thi? — **aksar masla harness ka hota hai.**

### Designing the Loop
Model call → tool calls → results/errors **wapas** model ko → repeat → step/time/token limit par rukna. Errors chhupao mat — wazeh error message model ke liye **feedback** hai.

### Harness Enforces vs Model Decides
| Model decide kare | Harness enforce kare |
|---|---|
| Agla tool kaunsa | Kaunse tools **mojood** hain, permissions |
| Code kaise likhna | Test files edit nahi ho sakti |
| Kab "done" kehna | "Done" tabhi qabool jab **tests pass** |
| Plan | Step/token/cost budget, risky action par **human approval**, safe defaults |

### Many Context Windows (lambe kaam)
Kaam ek window mein nahi samata. Pattern (Anthropic ki long-running agent guidance isi tarah ka hai):
1. **Initializer run:** spec parh kar **task list** (JSON) aur progress file banao
2. **Worker runs:** har run **fresh context** se shuru — `AGENTS.md` + task list + progress notes parho → **ek** task khatam → verify → progress update → commit
3. Agla run wahin se jahan pichla chhora

### Durable Artifacts = Sachai ka source
Progress files, task lists, git commits aur tests — **conversation history nahi**. History compact/reset ho sakti hai; files nahi.

### Self-Run Feedback Loops
Agent khud tests, linters, type checks chalaye, output parhe, theek kare — **phir** done. (L0.6 Verification Loop ka automated version.)

### Agent-Readable Environment
`AGENTS.md`/`CLAUDE.md` jaisi **chhoti** instruction file: project structure, commands, qawaid. Saaf folder structure, docs. Agent ko andaze nahi lagane parne chahiye.

### Sub-Agents for Isolated Context
Focused subtask (code review, research, file summary) ko **naya saaf context**; wapas sirf **condensed nateeja** (≤100 words). Main agent ka context saaf rehta hai. Trade-off: extra calls, coordination; sub-agent ko poora context nahi pata.

## 💻 Code — Mini Coding-Agent Harness (end-to-end)
Ek chhota Python project jis mein 3 functions implement karne hain. Harness:
- `AGENTS.md`, `SPEC.md`, stub code aur **tests** (jo agent edit nahi kar sakta)
- **Initializer run** → `tasks.json` (harness validate karta hai)
- **Worker runs** — har run fresh context, ek task, `PROGRESS.md` update
- `mark_task_done` **khud tests chalata hai** — fail to reject (harness enforces)
- Path sandbox, protected files, `delete_file` par approval gate
- `ask_reviewer` **sub-agent** (isolated context)
- Per-run step aur token budget, observation masking

In [ ]:
PROJ = WORK / "harness_project"
if PROJ.exists():
    shutil.rmtree(PROJ)
PROJ.mkdir(parents=True)

FILES = {
"AGENTS.md": """# AGENTS.md — project guide for coding agents
## Structure
- textutils.py      → implementation (yahan code likho)
- test_textutils.py → tests (PROTECTED — edit mana hai)
- SPEC.md           → requirements
- tasks.json        → task list (harness manage karta hai)
- PROGRESS.md       → har run ke baad 1-2 lines notes
## Commands
- run_tests(pattern) → unittest; pattern jaise 'slugify'
## Rules
1. Ek run mein sirf EK task. 2. Pehle read_file se code dekho, phir write_file (poori file likho).
3. mark_task_done se pehle run_tests. 4. Sirf standard library. 5. PROGRESS.md mein note likho (update_progress).
""",
"SPEC.md": """# textutils spec
1. slugify(text) -> str: lowercase; har non-alphanumeric sequence ko ek '-' se badlo; shuru/aakhir ke '-' hatao.
   slugify('Hello, World!') == 'hello-world'; slugify('  Diver Pro 200m ') == 'diver-pro-200m'
2. word_count(text) -> dict: lowercase words (sirf a-z0-9) ki frequency. word_count('A a b!') == {'a': 2, 'b': 1}
3. truncate(text, max_len, suffix='...') -> str: agar len(text) <= max_len to text waisa hi.
   Warna text[:max_len - len(suffix)].rstrip() + suffix.  truncate('Hello world', 8) == 'Hello...'
""",
"textutils.py": """def slugify(text):
    raise NotImplementedError


def word_count(text):
    raise NotImplementedError


def truncate(text, max_len, suffix="..."):
    raise NotImplementedError
""",
"test_textutils.py": """import unittest
from textutils import slugify, word_count, truncate

class TestSlugify(unittest.TestCase):
    def test_slugify_basic(self):
        self.assertEqual(slugify("Hello, World!"), "hello-world")
    def test_slugify_spaces(self):
        self.assertEqual(slugify("  Diver Pro 200m "), "diver-pro-200m")
    def test_slugify_symbols(self):
        self.assertEqual(slugify("--Rs 9,500 / only!!"), "rs-9-500-only")

class TestWordCount(unittest.TestCase):
    def test_word_count_basic(self):
        self.assertEqual(word_count("A a b!"), {"a": 2, "b": 1})
    def test_word_count_empty(self):
        self.assertEqual(word_count(""), {})

class TestTruncate(unittest.TestCase):
    def test_truncate_short(self):
        self.assertEqual(truncate("Hi", 5), "Hi")
    def test_truncate_long(self):
        self.assertEqual(truncate("Hello world", 8), "Hello...")
    def test_truncate_suffix(self):
        self.assertEqual(truncate("abcdefghij", 6, suffix="~"), "abcde~")

if __name__ == "__main__":
    unittest.main()
""",
"PROGRESS.md": "# Progress log\n",
}
for name, content in FILES.items():
    (PROJ / name).write_text(content, encoding="utf-8")
print("Project ready:", sorted(p.name for p in PROJ.iterdir()))

In [ ]:
APPROVAL_MODE = "deny"     # "deny" = risky actions khud reject | "ask" = input() se aap se poochega

class MiniHarness:
    PROTECTED = {"test_textutils.py", "AGENTS.md", "SPEC.md", "tasks.json"}

    def __init__(self, root, model=None, max_steps_per_run=14, token_budget_per_run=60_000):
        self.root = Path(root).resolve()
        self.model = model or SMART_MODEL
        self.max_steps, self.token_budget = max_steps_per_run, token_budget_per_run
        self.run_id, self.current_task, self.task_done = 0, None, False
        self.events = []

    # ---------- guardrails ----------
    def _safe_path(self, path):
        p = (self.root / path).resolve()
        if self.root != p and self.root not in p.parents:
            raise PermissionError(f"path '{path}' project folder se bahar hai")
        return p

    def _approve(self, action):
        self.events.append(f"APPROVAL? {action}")
        if APPROVAL_MODE == "ask":
            return input(f"Approve '{action}'? (y/n) ").strip().lower() == "y"
        return False

    # ---------- tools ----------
    def list_files(self):
        return {"files": sorted(p.name for p in self.root.iterdir() if p.is_file())}

    def read_file(self, path):
        p = self._safe_path(path)
        if not p.exists():
            return {"error": f"{path} mojood nahi"}
        txt = p.read_text(encoding="utf-8")
        return txt if len(txt) < 6000 else txt[:6000] + "\n...[truncated]"

    def write_file(self, path, content):
        p = self._safe_path(path)
        if p.name in self.PROTECTED:
            return {"error": f"{p.name} PROTECTED hai — harness ne write block kiya"}
        p.write_text(content, encoding="utf-8")
        return {"written": p.name, "chars": len(content)}

    def delete_file(self, path):
        p = self._safe_path(path)
        if not self._approve(f"delete {p.name}"):
            return {"error": "human approval nahi mili — delete cancel"}
        p.unlink(missing_ok=True)
        return {"deleted": p.name}

    def run_tests(self, pattern: str = None):
        cmd = [sys.executable, "-m", "unittest", "-v"] + (["-k", pattern] if pattern else [])
        try:
            res = subprocess.run(cmd, cwd=self.root, capture_output=True, text=True, timeout=60)
        except subprocess.TimeoutExpired:
            return {"passed": False, "output": "TIMEOUT (60s)"}
        out = (res.stdout + res.stderr)[-1800:]
        return {"passed": res.returncode == 0, "output": out}

    def update_progress(self, note):
        with open(self.root / "PROGRESS.md", "a", encoding="utf-8") as f:
            f.write(f"- run {self.run_id} [{self.current_task['id'] if self.current_task else '-'}]: {note.strip()[:300]}\n")
        return {"ok": True}

    def mark_task_done(self, task_id):
        tasks = self._tasks()
        t = next((x for x in tasks if x["id"] == task_id), None)
        if t is None:
            return {"error": f"task {task_id} nahi mila"}
        if self.current_task and task_id != self.current_task["id"]:
            return {"error": f"is run ka task {self.current_task['id']} hai, {task_id} nahi"}
        result = self.run_tests(t["test_pattern"])                # HARNESS khud verify karta hai
        if not result["passed"]:
            self.events.append(f"REJECTED done for {task_id}")
            return {"error": "tests FAIL — task done nahi mana gaya. Output parho aur theek karo.", "output": result["output"][-900:]}
        t["status"] = "done"
        self._save_tasks(tasks)
        self.task_done = True
        self.events.append(f"DONE {task_id}")
        self._git_commit(f"{task_id}: {t['title']}")
        return {"ok": True, "message": f"{task_id} done ✅ (tests passed)"}

    def ask_reviewer(self, path, question):
        """SUB-AGENT: alag, saaf context. Main agent ko sirf mukhtasar jawab milta hai."""
        content = self.read_file(path)
        r = llm([{"role": "system", "content": "Tum code reviewer ho. Jawab 80 words se kam. Sirf aham baat."},
                 {"role": "user", "content": f"File {path}:\n```\n{content}\n```\nSawal: {question}"}],
                model=FAST_MODEL, tag=f"6.8-subagent-run{self.run_id}", max_tokens=1200)
        return {"review": r["text"][:700]}

    # ---------- durable state ----------
    def _tasks(self):
        p = self.root / "tasks.json"
        return json.loads(p.read_text(encoding="utf-8")) if p.exists() else []

    def _save_tasks(self, tasks):
        (self.root / "tasks.json").write_text(json.dumps(tasks, ensure_ascii=False, indent=1), encoding="utf-8")

    def _git_commit(self, msg):
        if shutil.which("git") is None:
            return
        try:
            if not (self.root / ".git").exists():
                subprocess.run(["git", "init", "-q"], cwd=self.root, check=False)
            subprocess.run(["git", "add", "-A"], cwd=self.root, check=False)
            subprocess.run(["git", "-c", "user.name=harness", "-c", "user.email=h@local", "commit", "-qm", msg],
                           cwd=self.root, check=False, capture_output=True)
        except Exception:
            pass

    def tool_defs(self):
        S = lambda **p: {k: {"type": v} for k, v in p.items()}
        return [
            tool_schema("list_files", "Project ki files."),
            tool_schema("read_file", "File parho.", S(path="string"), ["path"]),
            tool_schema("write_file", "Poori file likho (overwrite). Protected files block hain.", S(path="string", content="string"), ["path", "content"]),
            tool_schema("delete_file", "File delete (human approval chahiye).", S(path="string"), ["path"]),
            tool_schema("run_tests", "Unittest chalao; pattern se filter (jaise 'slugify').", S(pattern="string")),
            tool_schema("update_progress", "PROGRESS.md mein 1-2 line note.", S(note="string"), ["note"]),
            tool_schema("mark_task_done", "Task ko done mark karo — harness tests chala kar verify karega.", S(task_id="string"), ["task_id"]),
            tool_schema("ask_reviewer", "Sub-agent se file ka review/sawal (alag context).", S(path="string", question="string"), ["path", "question"]),
        ]

    def registry(self):
        return {n: getattr(self, n) for n in ["list_files", "read_file", "write_file", "delete_file", "run_tests",
                                              "update_progress", "mark_task_done", "ask_reviewer"]}

    # ---------- runs ----------
    def initializer_run(self):
        test_names = re.findall(r"def (test_\w+)", (self.root / "test_textutils.py").read_text(encoding="utf-8"))
        prompt = (f"SPEC:\n{(self.root / 'SPEC.md').read_text(encoding='utf-8')}\n\nTest names: {test_names}\n\n"
                  'Task list JSON do: {"tasks": [{"id": "T1", "title": str, "test_pattern": str}]}. '
                  "Har function ka ek task. test_pattern aisa substring ho jo us function ke tests ke naam mein ho.")
        r = llm(prompt, model=self.model, json_mode=True, tag="6.8-init", max_tokens=1500)
        try:
            tasks = json.loads(r["text"])["tasks"]
            assert tasks and all(any(t["test_pattern"] in n for n in test_names) for t in tasks)   # harness validation
            source = "LLM"
        except Exception as e:
            self.events.append(f"init LLM output invalid ({e}) → fallback")
            tasks = [{"id": f"T{i + 1}", "title": f"implement {f}", "test_pattern": f} for i, f in enumerate(["slugify", "word_count", "truncate"])]
            source = "fallback"
        for t in tasks:
            t["status"] = "pending"
        self._save_tasks(tasks)
        print(f"🧭 initializer ({source}): {[(t['id'], t['test_pattern']) for t in tasks]}")

    def worker_run(self):
        self.run_id += 1
        tasks = self._tasks()
        self.current_task = next((t for t in tasks if t["status"] == "pending"), None)
        if not self.current_task:
            return False
        self.task_done = False
        tag = f"6.8-worker-run{self.run_id}"
        progress_tail = "".join((self.root / "PROGRESS.md").read_text(encoding="utf-8").splitlines(True)[-6:])
        system = ((self.root / "AGENTS.md").read_text(encoding="utf-8") +
                  f"\n## Current tasks\n{json.dumps([{k: t[k] for k in ('id', 'title', 'status')} for t in tasks])}"
                  f"\n## Recent progress\n{progress_tail}")
        user = (f"Is run ka task: {self.current_task['id']} — {self.current_task['title']} "
                f"(tests pattern: '{self.current_task['test_pattern']}'). SPEC.md aur textutils.py parho, implement karo, "
                "run_tests chalao, update_progress likho, phir mark_task_done.")
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]

        def on_step(messages, step):
            mask_old_tool_results(messages, keep_last=3)                  # observation build-up control
            if self.task_done:
                return "[task done — run khatam]"
            used = usage_summary(tag).get("prompt_tokens", 0)
            if used > self.token_budget:
                self.events.append(f"run {self.run_id}: token budget exceeded ({used})")
                return "[token budget khatam]"
            return None

        print(f"\n▶️ RUN {self.run_id} | task {self.current_task['id']} | fresh context = {count_msg_tokens(msgs)} tokens")
        final, msgs, steps = run_tool_loop(msgs, self.tool_defs(), self.registry(), model=self.model,
                                           max_steps=self.max_steps, on_step=on_step, tag=tag, max_tokens=3000)
        print(f"   ⏹ {final[:80]} | steps={steps} | {usage_summary(tag)}")
        return True

    def run(self, max_runs=6):
        if not (self.root / "tasks.json").exists():
            self.initializer_run()
        while self.run_id < max_runs and any(t["status"] == "pending" for t in self._tasks()):
            self.worker_run()
        return self._tasks()

harness = MiniHarness(PROJ)
final_tasks = harness.run(max_runs=6)
print("\nFinal tasks:", [(t["id"], t["status"]) for t in final_tasks])
print("Full test suite:", "PASS ✅" if harness.run_tests()["passed"] else "FAIL ❌")
print("Harness events:", harness.events)

In [ ]:
print("=== PROGRESS.md (durable artifact) ===")
print((PROJ / "PROGRESS.md").read_text(encoding="utf-8"))
print("=== textutils.py (agent ka likha code) ===")
print((PROJ / "textutils.py").read_text(encoding="utf-8"))

**Gaur karein:**
1. Har worker run **fresh context** se shuru hua (sirf ~1k tokens) — pichli runs ki baatcheet nahi, sirf `AGENTS.md` + `tasks.json` + `PROGRESS.md`. Yahi "many context windows" pattern hai.
2. Agar model ne tests fail hone par `mark_task_done` kiya to harness ne **reject** kiya (events dekhein) — model "done" ka faisla nahi karta, harness karta hai.
3. Test file edit karne ki koshish block hoti hai (reward hacking se bachao — L13.6 yaad karein).
4. **Exercise:** `APPROVAL_MODE = "ask"` kar ke model se kahein "PROGRESS.md delete karo" — approval gate dekhein.

### 🎤 Interview angle (6.8)
- *"Same model, aap ka agent kharab chal raha hai aur Claude Code achha — kyun?"* → harness: tools ki quality, context management, verification loops, error feedback, permissions, AGENTS.md, sub-agents.
- *"Agent ne kaha 'done' magar kaam adhoora tha"* → done ko **verifiable** banao (tests/checks harness chalaye), model ke kehne par nahi.
- *"Kayi ghante ka task kaise chalaoge?"* → initializer + worker runs, task list JSON, progress file, git commits, har run ek task, fresh context, resume.

✅ **Outcome:** Harness jo agent ko itne lambe kaam par productive rakhe jo ek context window mein na samaye.

---
# 🚀 Capstone · Context-Managed Long-Horizon Agent
*Roadmap ki shartain: explicit context budget · history compaction · external state · selective retrieval · tool-output compression · harness (progress + verification + limits) · goal/constraints poore run mein mehfooz · naive vs managed ka naap tol.*

### Kaam
Support inbox mein **N tickets** hain. Har ticket lamba hai (metadata, signature, legal disclaimer, purani history — asli emails ki tarah faltu maloomat se bhara). Agent ko:
1. Har ticket ki **category** (`refund` / `exchange` / `delivery_delay` / `warranty`) pehchanni hai
2. Refund tickets ki **refund raqam** nikalni hai (dhoka: tickets mein doosri raqamein bhi hain — product ki qeemat waghera)
3. Aakhir mein report: category counts, refund ticket IDs, total refund

### Do agents
| | **Naive agent** | **Managed agent** |
|---|---|---|
| History | Har ticket ka poora body context mein jama | Recorded tickets ke bodies **mask** (observation masking) |
| State | Model ke "dimagh" mein | **External state** (`record_finding` tool) — durable |
| Goal/constraints | Sirf pehle message mein | System mein pinned + har step **compact state view** |
| Tool output | Poora ticket (metadata, legal, signature sab) | Harness **boilerplate strip** karta hai (tool-output compression) |
| Aggregation | Model khud jor/gin kar likhta hai | **Harness code** se compute (deterministic) |
| "Done" | Model jab chahe | `finish` tabhi qabool jab **sab tickets recorded** (verification) |
| Limits | max_steps | max_steps + token budget |

In [ ]:
N_TICKETS = 10          # rate limit ho to 6-8 kar dein
CATS = ["refund", "exchange", "delivery_delay", "warranty"]
BOILER = ("--- TICKET METADATA --- channel=whatsapp | priority=normal | sla=24h | queue=support-pk | "
          "assigned=unassigned | tags=[] | created_via=webhook-v2 | locale=ur-PK\n"
          "Previous interactions: customer ne pichle saal 2 orders kiye the (dono delivered, koi complaint nahi). "
          "Loyalty tier: silver. Marketing opt-in: yes.\n"
          "--- LEGAL --- Yeh message aur is ke attachments confidential hain. Agar aap ghalti se receive karein to "
          "foran delete karein. TimeCraft PK kisi bhi delay ki zimmedari courier policy ke mutabiq leta hai. "
          "Customer data Pakistan mein store hota hai aur sirf order processing ke liye istemal hota hai.\n"
          "--- SIGNATURE --- Sent from my phone. Please reply jaldi. Shukriya, Allah Hafiz.\n")

def make_tickets(n, seed=21):
    rng = random.Random(seed)
    order = (CATS * ((n // 4) + 1))[:n]
    rng.shuffle(order)
    tickets = {}
    for i, cat in enumerate(order):
        tid = f"T-{101 + i}"
        p = rng.choice(PRODUCTS)
        if cat == "refund":
            amt = rng.choice([4500, 6500, 9500, 12500])
            msg = (f"Maine {p['name']} Rs {p['price']:,} mein li thi, magar strap toot gaya aur exchange nahi chahiye. "
                   f"Store ne partial settlement offer kiya hai — mujhe sirf Rs {amt:,} ka refund chahiye, bank transfer se.")
        elif cat == "exchange":
            amt = 0
            msg = f"Mujhe {p['name']} (Rs {p['price']:,}) ka rang pasand nahi aaya, black wale se exchange karna hai. Paise wapas nahi chahiye."
        elif cat == "delivery_delay":
            amt = 0
            msg = f"Mera order 9 din se nahi aaya, {p['name']} thi Rs {p['price']:,} wali. Courier tracking update nahi ho rahi."
        else:
            amt = 0
            msg = f"{p['name']} 3 mahine purani hai, ab time aage peeche ho raha hai. Warranty claim karna hai, Rs {p['price']:,} ki thi."
        tickets[tid] = {"category": cat, "refund": amt,
                        "body": f"TICKET {tid}\n{BOILER}\n--- CUSTOMER MESSAGE ---\n{msg}\n{BOILER}"}
    return tickets

TICKETS = make_tickets(N_TICKETS)
TRUTH = {"category_counts": {c: sum(1 for t in TICKETS.values() if t["category"] == c) for c in CATS},
         "refund_ticket_ids": sorted(k for k, t in TICKETS.items() if t["category"] == "refund"),
         "total_refund_pkr": sum(t["refund"] for t in TICKETS.values())}
print("Ticket body size:", count_tokens(TICKETS["T-101"]["body"]), "tokens each")
print("Ground truth:", TRUTH)

def list_tickets():
    return {"ticket_ids": list(TICKETS)}

def get_ticket(ticket_id: str):
    t = TICKETS.get(ticket_id)
    return t["body"] if t else {"error": f"{ticket_id} nahi mila"}

TASK = (f"Support inbox ke saare {N_TICKETS} tickets process karo. Har ticket ki category ({', '.join(CATS)}) pehchano; "
        "refund tickets mein customer jo REFUND raqam maang raha hai woh nikalo (product ki qeemat refund nahi hai).")

### Agent A — Naive (sab kuch context mein jama)

In [ ]:
NAIVE_SYS = (TASK + "\nTools: list_tickets, get_ticket. Jab sab ho jaye to aakhri jawab SIRF yeh JSON ho:\n"
             '{"category_counts": {"refund": n, "exchange": n, "delivery_delay": n, "warranty": n}, '
             '"refund_ticket_ids": [...], "total_refund_pkr": n}')
naive_tools = [tool_schema("list_tickets", "Saare ticket IDs."),
               tool_schema("get_ticket", "Ek ticket ka poora text.", {"ticket_id": {"type": "string"}}, ["ticket_id"])]

t0 = time.time()
naive_msgs = [{"role": "system", "content": NAIVE_SYS}, {"role": "user", "content": "Shuru karo."}]
try:
    naive_final, naive_msgs, naive_steps = run_tool_loop(
        naive_msgs, naive_tools, {"list_tickets": list_tickets, "get_ticket": get_ticket},
        model=FAST_MODEL, max_steps=N_TICKETS + 6, tag="cap-naive", verbose=False, max_tokens=2500)
except RuntimeError as e:          # context itna bara ho gaya ke request hi reject — naive approach ki asli naakami
    print("💥 Naive agent crash:", e)
    naive_final, naive_steps = "", -1

def parse_report(text):
    m = re.search(r"\{.*\}", text or "", re.S)
    try:
        return json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        return {}

naive_report = parse_report(naive_final)
print(f"steps={naive_steps} | wall={time.time() - t0:.1f}s | final context ≈ {count_msg_tokens(naive_msgs):,} tokens")
print("Naive report:", naive_report)

### Agent B — Managed (budget · external state · masking · verification)

In [ ]:
class ManagedTicketAgent:
    def __init__(self, model=None, token_budget=80_000, max_steps=N_TICKETS * 3 + 6):
        self.model, self.token_budget, self.max_steps = model or FAST_MODEL, token_budget, max_steps
        self.state = {}                      # ticket_id -> {"category", "refund_amount_pkr"}  (DURABLE STATE)
        self.state_file = WORK / "capstone_state.json"
        self.finished, self.report, self.rejections = False, None, 0

    # ----- tools -----
    def record_finding(self, ticket_id: str, category: str, refund_amount_pkr: int = 0):
        if ticket_id not in TICKETS:
            return {"error": f"unknown ticket {ticket_id}"}
        if category not in CATS:
            return {"error": f"category {CATS} mein se ho"}
        try:
            amt = int(str(refund_amount_pkr).replace(",", "") or 0)
        except ValueError:
            return {"error": "refund_amount_pkr number ho"}
        if category != "refund":
            amt = 0
        self.state[ticket_id] = {"category": category, "refund_amount_pkr": amt}
        self.state_file.write_text(json.dumps(self.state, indent=1), encoding="utf-8")   # offload to disk
        return {"recorded": ticket_id, "processed": len(self.state), "remaining": N_TICKETS - len(self.state)}

    def finish(self, note: str = ""):
        missing = [t for t in TICKETS if t not in self.state]
        if missing:                                                    # HARNESS VERIFICATION
            self.rejections += 1
            return {"error": f"finish REJECT — {len(missing)} tickets baqi: {missing}"}
        cc = {c: sum(1 for v in self.state.values() if v["category"] == c) for c in CATS}
        self.report = {"category_counts": cc,                          # deterministic aggregation (code, not LLM)
                       "refund_ticket_ids": sorted(k for k, v in self.state.items() if v["category"] == "refund"),
                       "total_refund_pkr": sum(v["refund_amount_pkr"] for v in self.state.values()),
                       "agent_note": note[:300]}
        self.finished = True
        return {"ok": True, "report": self.report}

    # ----- context management -----
    @staticmethod
    def get_ticket_compressed(ticket_id: str):
        """Tool-output compression: harness faltu metadata/legal/signature hata kar sirf kaam ka hissa deta hai."""
        body = get_ticket(ticket_id)
        if not isinstance(body, str):
            return body
        m = re.search(r"--- CUSTOMER MESSAGE ---\n(.*?)\n--- TICKET METADATA", body, re.S)
        return f"TICKET {ticket_id}\n{m.group(1).strip() if m else body[:600]}"

    def state_view(self):
        remaining = [t for t in TICKETS if t not in self.state]
        return (f"[[STATE]] processed={len(self.state)}/{N_TICKETS} | remaining={remaining} | "
                f"findings={json.dumps(self.state)}\nYaad rakho: goal = {TASK}")

    def on_step(self, messages, step):
        # (1) tool_call_id → ticket mapping, phir sirf RECORDED tickets ke bodies mask
        call_ticket = {}
        for m in messages:
            for tc in (m.get("tool_calls") or []):
                if tc["function"]["name"] == "get_ticket":
                    try:
                        call_ticket[tc["id"]] = json.loads(tc["function"]["arguments"]).get("ticket_id")
                    except Exception:
                        pass
        for m in messages:
            if m.get("role") == "tool" and call_ticket.get(m.get("tool_call_id")) in self.state and not m["content"].startswith("[masked"):
                m["content"] = f"[masked: {call_ticket[m['tool_call_id']]} recorded in external state]"
        # (2) purana state message hatao, naya compact state view aakhir mein (goal/constraints mehfooz)
        messages[:] = [m for m in messages if not (m.get("role") == "user" and str(m.get("content", "")).startswith("[[STATE]]"))]
        messages.append({"role": "user", "content": self.state_view()})
        # (3) limits
        if self.finished:
            return "[finished — verified]"
        if usage_summary("cap-managed").get("prompt_tokens", 0) > self.token_budget:
            return "[token budget khatam]"
        return None

    def run(self):
        sys_prompt = (TASK + "\nWorkflow: list_tickets → get_ticket → record_finding (refund na ho to refund_amount_pkr=0). "
                      "Steps bachane ke liye ek hi step mein pichle ticket ka record_finding AUR agle ticket ka get_ticket "
                      "saath (parallel) bhej sakte ho. Sab record ho jayein to finish.")
        tools = [naive_tools[0], naive_tools[1],
                 tool_schema("record_finding", "Ek ticket ka nateeja durable state mein save karo.",
                             {"ticket_id": {"type": "string"}, "category": {"type": "string", "enum": CATS},
                              "refund_amount_pkr": {"type": "integer"}}, ["ticket_id", "category", "refund_amount_pkr"]),
                 tool_schema("finish", "Sab tickets record hone ke baad chalao. Harness verify karega.", {"note": {"type": "string"}})]
        reg = {"list_tickets": list_tickets, "get_ticket": self.get_ticket_compressed,
               "record_finding": self.record_finding, "finish": self.finish}
        msgs = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": "Shuru karo."}]
        final, msgs, steps = run_tool_loop(msgs, tools, reg, model=self.model, max_steps=self.max_steps,
                                           on_step=self.on_step, tag="cap-managed", verbose=False, max_tokens=2500)
        return final, msgs, steps

t0 = time.time()
mgr = ManagedTicketAgent()
m_final, m_msgs, m_steps = mgr.run()
print(f"steps={m_steps} | wall={time.time() - t0:.1f}s | final context ≈ {count_msg_tokens(m_msgs):,} tokens | finish rejections={mgr.rejections}")
print("Managed report:", mgr.report)

### 📊 Muqabla — coherence, tokens, latency, cost

In [ ]:
def grade(report):
    if not report:
        return {"counts_ok": False, "refund_ids_ok": False, "total_ok": False}
    return {"counts_ok": report.get("category_counts") == TRUTH["category_counts"],
            "refund_ids_ok": sorted(report.get("refund_ticket_ids", [])) == TRUTH["refund_ticket_ids"],
            "total_ok": report.get("total_refund_pkr") == TRUTH["total_refund_pkr"]}

cmp = {"naive": {**usage_summary("cap-naive"), **grade(naive_report)},
       "managed": {**usage_summary("cap-managed"), **grade(mgr.report)}}
keys = ["calls", "prompt_tokens", "peak_prompt", "completion_tokens", "latency_s", "cost_usd", "counts_ok", "refund_ids_ok", "total_ok"]
print(f"{'metric':<20}{'naive':>14}{'managed':>14}")
for k in keys:
    print(f"{k:<20}{str(cmp['naive'].get(k, '-')):>14}{str(cmp['managed'].get(k, '-')):>14}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for tag, lbl in [("cap-naive", "naive"), ("cap-managed", "managed")]:
    axes[0].plot([r["prompt_tokens"] for r in USAGE_LOG if r["tag"] == tag], marker=".", label=lbl)
axes[0].set_title("Har LLM call ka prompt size"); axes[0].set_xlabel("call #"); axes[0].set_ylabel("prompt tokens"); axes[0].legend()
axes[1].bar(["naive", "managed"], [cmp["naive"].get("prompt_tokens", 0), cmp["managed"].get("prompt_tokens", 0)], color=["#c0392b", "#27ae60"])
axes[1].set_title("Kul input tokens (bill)"); plt.tight_layout(); plt.show()

### 🧾 Capstone ka nateeja kaise parhein (aur interview mein kaise sunayein)
- **Naive** mein har call ka prompt size **seedhi line mein barhta** hai (har ticket body jama) — kul tokens quadratic ki taraf jate hain. Model ko aakhir mein 10 tickets "yaad" kar ke ginna parta hai → ghalti ka imkaan (khaas kar distractor raqamein).
- **Managed** mein prompt size **bounded** rehta hai (boilerplate strip + masking + compact state), facts durable state mein, aggregation code se, aur `finish` verification adhoora kaam nahi hone deta.
- Ghaur karein: managed agent ke **calls zyada** ho sakte hain (record_finding ki wajah se) — phir bhi har call chhota hai. Isi liye "kul tokens" aur "peak context" dono naapne zaroori hain; sirf calls ginna gumrah karta hai.
- Interview story format (L15.8): *"Naive approach ka peak context X tokens tha aur accuracy Y; maine observation masking + external state + verified finish lagaya, peak Z tokens, cost N% kam, accuracy 100%."* — **numbers ke saath.**

**Mazeed experiments (khud karein):**
1. `N_TICKETS = 30` — naive kab tootta hai? (rate limit / context / accuracy)
2. Managed agent mein masking band karein (on_step ka step (1) comment) — tokens par asar naapein
3. `SMART_MODEL` vs `FAST_MODEL` — accuracy vs cost

---
# 🎤 Interview Prep — 22 sawal, model jawab

> Jawab dene ka tareeqa (L15.9): **definition → kyun zaroori → kaise (technique) → trade-off → misaal/number.**

**Q1. Context engineering kya hai aur prompt engineering se kaise mukhtalif hai?**
Context engineering har LLM call par window mein *sahi maloomat, sahi miqdaar, sahi tarteeb* dene ka nizaam hai — instructions, history, retrieved docs, memory, tool definitions/results aur state sab. Prompt engineering is ka ek hissa (hidayat likhna) hai. Prompt static hota hai; context har step par dynamically assemble hota hai, khaas kar agents mein.

**Q2. Jab 1M token ki window hai to context manage karne ki zaroorat kyun?**
(1) Har request par poore context ka kharcha aur latency, (2) context rot — lambai ke saath accuracy girti hai, (3) lost-in-the-middle, (4) distractors/conflicts se confusion, (5) access control aur freshness — sab kuch har user ko nahi dikhana. Bari window gunjaish hai, strategy nahi.

**Q3. Context rot kya hai? Kaise naapoge?**
Input lamba hone par performance ka girna, window bharne se pehle hi. Naap: fixed task (needle with distractors, multi-fact aggregation) ko barhti context lambai par chalao, accuracy vs tokens plot karo (6.1 jaisa experiment). Apne asli data par karo, benchmark par nahi.

**Q4. Context ke 4 failure modes batao.**
Poisoning (ghalat baat context mein aa kar baar baar refer hoti hai), Distraction (lambi history model ko purane patterns dohrane par laati hai), Confusion (faltu tools/docs se ghalat chunaav), Clash (context ke hisse aapas mein takrate hain). Ilaj: validation + state reset, compaction, selection, stale/conflicting cheezen hatana + source priority.

**Q5. Write / Select / Compress / Isolate samjhao.**
Write = context se bahar save (notes, files, DB). Select = sirf mutalliqa andar laana (RAG, tool selection, memory retrieval). Compress = jo andar hai chhota karna (summaries, compaction, masking). Isolate = kaam alag contexts mein baantna (sub-agents, sandbox, state fields).

**Q6. Lost in the middle kya hai aur ilaj?**
Models lambe context ke shuru/aakhir ki maloomat behtar istemal karte hain, darmiyan ki kam (U-curve). Ilaj: retrieval se chhota context, reranking, sab se relevant chunks kinaron par (reorder), sawal aakhir mein, structure/tags.

**Q7. 200+ turns wale support chatbot ki memory design karo.**
Aakhri K turns verbatim + structured rolling summary (goal, constraints, decisions, open tasks) + pinned exact facts (IDs, raqam — regex/DB se) + long-term user memory store (preferences) jo zaroorat par retrieve ho. Summary ko threshold par batch mein update karo (cache-friendly). Probe-question eval se information loss naapo.

**Q8. Summarization mein kya khota hai aur kaise bachaoge?**
Exact numbers/IDs, negations, kis ne kya kaha, shartain, adhoore kaam. Bachao: structured schema, deterministic extraction of exact data, "kya mehfooz rakhna hai" ki explicit hidayat, aur probe eval (6.3).

**Q9. Compaction, summarization aur observation masking mein farq?**
Masking: purane tool results ko placeholder se badalna (sasta, deterministic, maloomat handle se wapas). Deterministic compaction: fields/rows kam karna. Summarization: LLM se khulasa (mehnga, lossy, extra latency). Pehle sasta, zaroorat par mehnga.

**Q10. Agent ke paas 80 tools hain aur accuracy gir rahi hai — kya karoge?**
Pehle naapo: tool definitions ke tokens aur tool-selection accuracy (eval). Phir: overlapping tools merge, saaf naam/descriptions, namespaces/groups, intent router ya tool retrieval se top-k exposure, stage-based tools (read phase vs write phase), sub-agents ko domain-wise tools. Phir dobara eval.

**Q11. Ek tool 50k tokens ka JSON deta hai. Kya karoge?**
Harness level par: pagination/filters, sirf zaroori fields, summary + `result_id` handle, aur ek `query_result` tool taake model detail lazily mangwaye. Raw output kabhi seedha context mein nahi. Output size cap.

**Q12. Prompt caching ke liye prompt kaise structure karoge? Cache kya todta hai?**
Stable → dynamic: system rules, fixed-order tools, static docs/examples, append-only history, aakhir mein naya message. Cache todne wale: shuru mein timestamps/request IDs, tools ki tarteeb badalna, random JSON key order, history ko beech se edit karna. `cached_tokens` se hit rate monitor.

**Q13. Cost: 10k requests/din, 4k static prefix, 600 dynamic, 300 output, $0.15/$0.60 per 1M. Caching (90% hit, 50% off) se kitni bachat?**
Baghair: per request (4600×0.15 + 300×0.60)/1e6 = $0.00087 → ~$261/maah. Caching ke saath cached = 3600 tokens par aadhi qeemat → input (1000×0.15 + 3600×0.075)/1e6 = $0.00042, +$0.00018 output = $0.0006 → ~$180/maah. (6.7 calculator se verify karein.) Prefix ko 1500 tak trim karein to aur kam.

**Q14. Agent harness kya hai? Same model mukhtalif nateeje kyun deta hai?**
Model ke gird loop, system prompt, tools, context management, state, permissions, verification, environment. Harness tay karta hai model ko kya dikhta hai, errors kaise wapas aate hain, "done" kaise verify hota hai — is liye same model alag harness mein bilkul alag perform karta hai.

**Q15. Ek kaam jo ek context window mein nahi samata — kaise design karoge?**
Initializer run: spec → task list (JSON) + progress file. Worker runs: fresh context, AGENTS.md + task list + progress notes parho, ek task karo, tests chalao, progress update, commit. Durable artifacts sachai ka source; conversation history disposable.

**Q16. Sub-agents kab use karoge? Nuqsan?**
Jab subtask focused ho aur bohot sa intermediate context paida kare (research, code search, review) — sub-agent saaf context mein kaam kar ke sirf khulasa lautaye. Nuqsan: extra calls/latency/cost, coordination, sub-agent ko poora context nahi pata, nateejon ka takrao. Specialization ka faida na ho to ek agent behtar (L5.6).

**Q17. Memory, offloading aur RAG mein farq?**
RAG: bahar ki knowledge (docs) par grounding. Memory: user/conversation ki durable baatein sessions ke darmiyan. Offloading: isi task ka working data bahar rakh kar window khali karna. Teeno mein "bahar store + zaroorat par retrieve" hai, maqsad alag.

**Q18. Debugging scenario: agent 40 steps ke baad bhatak jata hai aur same tools dohrata hai.**
Trace/window inspect karo: context kitna bara, kis cheez ne bhara (tool outputs?), goal abhi window mein hai ya dab gaya? Ilaj: observation masking, goal/plan ko har step compact state mein dobara dikhana, progress external state mein, loop detection (same tool+args dohrane par harness rokay), step budget, zaroorat par fresh context se resume.

**Q19. 32k model ke liye context budget kaise allocate karoge?**
Mota: system 5-10%, tools 5-15%, memory 5%, retrieval 25-40%, history 15-25%, state 5%, output reserve 15-20% (reasoning model ho to zyada). Overflow par priority se kaato: history summarize → retrieval k kam → tools filter. Asal numbers eval se tune.

**Q20. Context quality ko evaluate kaise karoge?**
Probe questions (known answers) compression ke pehle/baad; needle/aggregation tests barhti lambai par; A/B: tokens, latency, cost vs task accuracy; agents mein task completion rate aur trajectory efficiency (L7.9); production mein prompt_tokens percentiles aur cache hit rate monitor.

**Q21. Context mein conflicting maloomat ho (purana vs naya policy) to?**
Best: ingestion par hi superseded docs hatao (`supersedes`/version metadata). Phir: har doc par date/source label, priority rule (official > blog, nayi > purani), jawab mein source cite karwao, conflict ho to user/insaan ko batao.

**Q22. Context engineering ka security se kya taluq?**
Retrieved docs aur tool outputs untrusted data hain (indirect prompt injection — L8.2). Unhein tags mein band karo, instructions se alag rakho; memory mein PII/secrets na likho; context mein utna hi do jitna zaroori (least privilege for context); cross-tenant data context mein kabhi na mile.

---
# 🧾 Cheat Sheet — L6 ek page mein

| Module | Ek line | Key technique | Code mein |
|---|---|---|---|
| 6.1 | Context = budget (paisa, latency, attention) | Inspect window, budget allocation, 4 failure modes | `inspect_context`, `ContextBudget` |
| 6.2 | Sab se chhota kaam ka context | Score = relevance+importance+freshness+trust, greedy pack, tags, stale cutting | `ContextBuilder` |
| 6.3 | Chhota karo, magar zaroori na khoye | Structured summary + pinned exact facts, rolling/hierarchical, compaction | `RollingMemory`, `extract_pinned_facts` |
| 6.4 | Bahar likho, zaroorat par laao | Note store tools, task state file, active vs stored | `NoteStore`, `TaskState` |
| 6.5 | Bari window ≠ hal | Retrieval, reorder to edges, map-reduce, strategy by question type | `choose_strategy`, `reorder_for_edges` |
| 6.6 | Tools bhi context hain | Dynamic exposure, result handles, observation masking | `select_tools`, `ResultStore`, `mask_old_tool_results` |
| 6.7 | Stable prefix = cache | Stable→dynamic order, measure `cached_tokens`, cost math | `monthly_cost`, `ResponseCache` |
| 6.8 | Model + harness = agent | Fresh context per run, task list, progress file, tests-gated done, sub-agents, approvals | `MiniHarness` |

### ❌ Aam ghaltiyan
- "Bari window wala model le lo" — masla hal nahi hota
- Poora tool output / poori history har dafa bhejna
- LLM summary par exact numbers ke liye bharosa
- System prompt ke shuru mein timestamp (cache khatam)
- Model ke "done" ko maan lena baghair verification
- Context changes ko baghair eval ke ship karna

### 🔧 Apne project par lagayein (exercise)
Agar aap koi WhatsApp/e-commerce support bot bana rahe hain:
1. Har customer ki conversation ke liye `RollingMemory` (aakhri 6 turns + structured summary + pinned order IDs)
2. Order lookup tools ke outputs `compact_records` se chhote, poora data handle ke peeche
3. System prompt + policies stable prefix mein, customer ka naya message aakhir mein → caching
4. Policies mein `date`/`supersedes` metadata → purani policy kabhi bot tak na pohanche
5. Probe-question eval set banayein (10 asli sawal + jawab) aur har change par chalayein → yahi **Level 7 (LLM Evaluations)** ka darwaza hai

In [ ]:
# Poore notebook ka hisaab
total = usage_summary("")
print("Poore notebook ka LLM usage:", total)
by_section = {}
for r in USAGE_LOG:
    sec = r["tag"].split("-")[0] if r["tag"] else "misc"
    by_section.setdefault(sec, [0, 0])
    by_section[sec][0] += r["prompt_tokens"] + r["completion_tokens"]
    by_section[sec][1] += cost_usd(r)
for sec, (tk, c) in by_section.items():
    print(f"{sec:<10} {tk:>9,} tokens   ${c:.5f}")

---
### ✅ Level 6 checklist — tab mukammal samjhein jab:
- [ ] Har module ka code chala kar nateeje khud parhe
- [ ] Capstone mein naive vs managed ke numbers apne alfaaz mein samjha sakein
- [ ] 22 interview sawalon mein se har ek ka jawab baghair dekhe 1-2 minute mein de sakein
- [ ] Kam az kam ek technique apne kisi asli project mein laga kar before/after naapa

**Agla level → L7 · LLM Evaluations** (yahan jo "probe eval" aur "naap tol" kiya, wahan us ko poora science banayenge).